<a href="https://colab.research.google.com/github/changhoon0807/bigdata/blob/main/dfm_nowcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip3 install pandas==1.1.5

SyntaxError: invalid syntax (<ipython-input-2-02aad451c3cd>, line 1)

In [ ]:
import numpy, pandas, scipy

ImportError: ignored

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print(f"numpy {numpy.__version__}, pandas {pandas.__version__}, scipy {scipy.__version__}")

NameError: ignored

In [ ]:
import itertools
from tqdm import tqdm
import pickle
from scipy.signal import lfilter
from numpy.linalg import inv, det, pinv, solve
from scipy.linalg import eigh, block_diag
from numpy import kron, log, eye, diag, zeros, ones, empty
from pandas import MultiIndex as MI
from collections import namedtuple
from pandas.tseries.offsets import MonthBegin, MonthEnd, QuarterEnd, QuarterBegin

import pandas as pd
import numpy as np

idx = pd.IndexSlice

import shutil
import os

from datetime import datetime, date, timedelta
import math

In [ ]:
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [ ]:
bcolors = [(33 / 256, 89 / 256, 103 / 256),
           (242 / 256, 220 / 256, 219 / 256),
           (218 / 256, 238 / 256, 243 / 256),
           (119 / 256, 119 / 256, 119 / 256),
           (251 / 256, 193 / 256, 94 / 256),
           (142 / 256, 186 / 256, 66 / 256),
           (192 / 256, 0 / 256, 0 / 256),    ]

In [ ]:
mkdir nowcasting

In [ ]:
cp -a /content/drive/MyDrive/nowcasting/* nowcasting/

# Setup

In [ ]:
DFModel = namedtuple("DFModel", "A C Q R Z_0 V_0")
# DFM 모형의 specification과 관련
# Model:
#    Y_t = C_t Z_t + e_t for e_t ~ N(0, R)
#    Z_t = A Z_{t-1} + mu_t for mu_t ~ N(0, Q)
# A: transition matrix
# C: observation matrix
# Q: covariance matrix for residuals for transition matrix
# R: covariance matrix for residuals of observation matrix
# Z_0: Initial value of state(factor).
# V_0: Initial value of covariance of factor state vector

DFMStructure = namedtuple("DFMStructure", "r p Rcon q")
# DFM 모형의 여러가지 설정을 보관
# 아래 def DFM_structure을 참조

DFMResult = namedtuple(
    "DFMResult", "LL y mean std normalized factors M threshold max_iter gamma X S spec blocks"
)
# LL: array of loglik,
# y: smoothed X,
# mean: mean of transformed X
# std: standard deviation of transformed X,
# normalzied: smoothed and normalized (transformed-normalized) X,
# factors: smooted factor,
# M: estimated Model parameters (A, C, Q, R, Z_0, V_0),
# par: parameters for EM estimation
# X: transformed-normalized (filtered) observed variables,
# S: Model structure (r, p, Rcon, q),
# spec: description of observed variables (freq, transformation type, name, etc.),
# blocks: description of block structure

optNaN = namedtuple("optNaN", "method k")
# 각종함수에서 옵션을 설정하는 namedtuple
# optNaN.method는 결측치를 처리하는 방법을 설정해주는 옵션
# if method=1: replace all the missing values using lfilter()
# if method=2: replace missing values after removing leading and closing zeros(a row is missing if >80% is NaN)
# if method=3: only remove rows with leading and closing zeros
# if method=4: remove rows with leading and closing zeros & replace missing values(a raw is missing if all are NaN)
# if method=5: replace missing values with spline() then runs filtering()
# optNaN.k는 lfilter()의 argument를 설정
# remNaNs_spline 등의 함수를 참조

SmoothedFactor = namedtuple("SmoothedFactor", "Plag P X_sm F")
# Smoothedfactor는 para_const()의 output임
# para_const()는 새로운 데이터를 입수하거나 기존 데이터가 수정되어서
# 전망을 업데이트하는 과정(News_DFM)에서 이용되며
# 칼만필터를 이용하여 missing variables를 기입(Fill in)하는 과정임
# plag: Smoothed factor covariance for transition matrix
# P: Smoothed factor covariance matrix
# X_sm: Smoothed data matrix
# F: Smoothed factors

UpdatedResult = namedtuple("UpdatedResult", "o n N A F W I M df_n")
# updatedREsult는 전망을 업데이트하는 함수인 News_DFM()의 output임
# o: Old nowcast
# n: New nowcast
# N: News for each data series
# A: Observed series release values
# F: Forecasted series values
# W: News weight
# I: Difference between observed and predicted series values ("innovation")
# M: pd.Series that displays which variables are newly imported for which month
# df_n: 새로운 observations(=X_new)에서의 NaN만 DFM으로 예측된 variable을 채워넣는 것임
# df_n은 머신러닝과의 데이터 공유를위한 데이터셋 생성의 일환

Target = namedtuple("Target", "v t")
# Target은 전망을 시작하기 전에 전망 목표변수, 전망 대상 분기 등을 설정하는 namedtuple
# v: 전망 대상 변수, GDP전망시 'N_gdp'로 설정
# t: 전망 대상 분기의 마지막 일(calendar day)


def DFM_structure(blocks):

    # Number of factors for each block
    r = pd.Series(np.ones(blocks.shape[1], dtype=int), index=blocks.columns)

    # Number of lags in autoregressive of factor
    p = 1

    # Contraints on the loadings of the quartrly variables
    Rcon = np.array([[2, -1, 0, 0, 0], [3, 0, -1, 0, 0], [2, 0, 0, -1, 0], [1, 0, 0, 0, -1]])

    q = np.zeros(4)

    return DFMStructure(r, p, Rcon, q)


def get_target_vintages(tq, last_week='2100-12-31', sample_yrs=15, display_vintages=True):
    '''
    Input : (t_var) forecast variable, (t_year, t_month) year and month of the forecast target quarter,
            (sample_yrs) number of years that are used for DFM estimation.
    Output: (target) namedtuple with target variable and quarter (%Y-%m-%d),
            (fvintage) vintage date for first nowcasting and last date for DFM estimation,
            (lvintage) vintage date for last nowcasting, close to the release date for flash estimate,
            (sample_start) first date of sample periods for DFM estimation,
            (vintages) date range from fvintage and lvintage with weekly frequency.'''

    t_year, t_month = tq.year, tq.month

    fvintage = second_friday(date(t_year, t_month, 15) - timedelta(days = 30 * 3))
    lvintage = fourth_friday(date(t_year, t_month, 15) + timedelta(days = 30 * 1))
    sample_start = second_friday(date(t_year, t_month, 15) - timedelta(days = 365 * sample_yrs + 30 * 1))
    vintages = [v.strftime('%Y-%m-%d') for v in pd.date_range(fvintage, lvintage, freq='7D')]

    vintages = [week for week in vintages if week <= last_week]

    if display_vintages:
        print("\nReal-time nowcasting is estimated for", *[date[5:] for date in vintages])

    return fvintage, lvintage, sample_start, vintages


# 해당 월의 두번째 금요일을 반환
def second_friday(tdate):
    """
    yyyy-mm-dd = second_friday(tdate)
    """
    # The 8th is the lowest second day in the month
    second = date(tdate.year, tdate.month, 8)
    # What day of the week is the 8th?
    w = second.weekday()
    # Friday is weekday 4
    if w != 4:
        # Replace just the day (of month)
        second = second.replace(day=(8 + (4 - w) % 7))
    return second.strftime("%Y-%m-%d")


# 해당 월의 세번째 금요일을 반환
def third_friday(tdate):
    """
    yyyy-mm-dd = second_friday(tdate)
    """
    # The 8th is the lowest second day in the month
    third = date(tdate.year, tdate.month, 15)
    # What day of the week is the 15th?
    w = third.weekday()
    # Friday is weekday 4
    if w != 4:
        # Replace just the day (of month)
        third = third.replace(day=(15 + (4 - w) % 7))
    return third.strftime("%Y-%m-%d")


# 해당 월의 네번째 금요일을 반환
def fourth_friday(tdate):
    """
    yyyy-mm-dd = second_friday(tdate)
    """
    # The 8th is the lowest second day in the month
    fourth = date(tdate.year, tdate.month, 22)
    # What day of the week is the 15th?
    w = fourth.weekday()
    # Friday is weekday 4
    if w != 4:
        # Replace just the day (of month)
        fourth = fourth.replace(day=(22 + (4 - w) % 7))
    return fourth.strftime('%Y-%m-%d')


def load_data(datafile, specfile, sample_start="1985-04-01"):

    spec, _ = load_spec(specfile)

    Z = pd.read_excel(datafile, index_col=0, engine="openpyxl")
    Z.index = pd.to_datetime(Z.index)
    Z = Z.asfreq("M")

    if Z.dropna(how="all", axis=0).shape[0] != Z.shape[0]:
        print("Data file has missing dates.")

    Z.index.names = ["Time"]
    Z = Z.loc[:, spec.index]
    T, N = Z.shape
    f2m = dict(zip(["m", "q", "sa", "a"], [1, 3, 6, 12]))  #
    f2a = dict(zip(["m", "q", "sa", "a"], [12, 4, 2, 1]))  #
    X = pd.DataFrame(np.nan, index=Z.index, columns=Z.columns)

    def sa_res(y):
        y_na = y.dropna().copy()
        y_mean = y_na.mean()
        mdum = pd.DataFrame(index=y_na.index)

        for i in range(12):
            mdum[i] = (mdum.index.month == i + 1) * 1

        res = y_na - mdum.dot(np.linalg.inv(mdum.T.dot(mdum))).dot(mdum.T).dot(y_na)
        res = res + y_mean
        res = res.reindex(y.index)
        return res

    # seasonal adj
    for var in Z.columns:
        sa = spec.loc[var, "sa"]
        if sa == 1:
            df = sa_res(Z.loc[:, var].dropna())
            Z.loc[:, var].loc[df.index] = df

    for var in Z.columns:
        formular = spec.loc[var, "Transformation"]
        freq = spec.loc[var, "Frequency"]
        if formular == "raw":  # Levels (No Transformation)
            X.loc[:, var] = Z.loc[:, var]
        elif formular == "chg":  # Change (Difference)
            X.loc[:, var] = Z.loc[:, var].diff(f2m[freq])
        elif formular == "ch1":  # YoY Change (Difference)
            X.loc[:, var] = Z.loc[:, var].diff(12)
        elif formular == "pch":  # Percent Change
            X.loc[:, var] = Z.loc[:, var].pct_change(f2m[freq], fill_method=None) * 100
        elif formular == "pc1":  # YoY Percent Change
            X.loc[:, var] = Z.loc[:, var].pct_change(12, fill_method=None) * 100
        elif formular == "pca":  # Percent Change (annualised)
            X.loc[:, var] = ((Z.loc[:, var].pct_change(f2m[freq], fill_method=None) + 1) ** f2a[freq] - 1) * 100
        elif formular == "log":  # Natural log
            X.loc[:, var] = np.log(Z.loc[:, var])
        else:
            print("Transformation formular not found.")

    raw = Z
    transformed = X.loc[sample_start:]
    mean = transformed.mean(skipna=True)
    std = transformed.std(skipna=True)
    trans_normalized = (transformed - mean) / std

    return transformed, mean, std, trans_normalized, raw


def load_spec(filename):

    # EM 알고리즘을 위한 옵션
    # 10e-(threshold) : 델타(전값-지금값)가 10e-(threshold) 보다 작으면 스탑
    # max_iter 최대 돌리는 횟수 충족시 스탑
    # new_param = gamma*new_param + (1-gamma)*old_param전값. gamma는 별로 영향이 없음

    # sample_yrs : DFM 모형 추정에 이용되는 샘플기간 설정

    # 변수별 블록 지정 및 변수변환
    spec = pd.read_excel(filename, sheet_name="spec", engine="openpyxl")

    # keep variables with Model == 1 and drop 'Model' column
    spec = spec.loc[spec.Model.eq(1)].drop("Model", axis=1)
    spec_sort = pd.DataFrame()

    for f in ["d", "w", "m", "q", "sa", "a"]:
        spec_sort = pd.concat([spec_sort, spec.loc[spec.Frequency.eq(f)]], axis=0)

    UnitsTransformed_dict = {
        "lin": "Levels (No Transformation)",
        "chg": "Change (Difference)",
        "ch1": "Year over Year Change (Difference)",
        "pch": "Percent Change",
        "pc1": "Year over Year Percent Change",
        "pca": "Percent Change (Annual Rate)",
        "cch": "Continuously Compounded Rate of Change",
        "cca": "Continuously Compounded Annual Rate of Change",
        "log": "Natural Log",
    }

    spec_sort["UnitsTransformed"] = spec_sort["Transformation"]
    spec_sort["UnitsTransformed"] = spec_sort["UnitsTransformed"].replace(
        UnitsTransformed_dict
    )
    spec_sort = spec_sort.set_index("SeriesID")
    Blocks = spec_sort.filter(regex="^Block", axis=1)
    Blocks.columns = [i for j, i in Blocks.columns.str.split("-").tolist()]
    spec_sort = spec_sort[spec_sort.columns.drop(list(spec_sort.filter(regex="Block")))]

    return spec_sort, Blocks

# dfm

In [ ]:
def dfm(trans_normalized, mean, std, S, spec, blocks, threshold=0.002, max_iter=500, gamma=0.5, suffix='0', plot_lik=False):
    """Reestimate DFM parameters until logliklihood converges or number of iteration reaches predetermined limit

    DFMResult(LL, converged, X_sm, X_mean, X_std, X_nsm, Z, M, par, y, S, spec, blocks) =
        dfm(y, X_mean, X_std, M, S, spec, blocks, par, filename=None)
    """
    last_time = datetime.now()

    filename = (f"/content/nowcasting/model/DFM{suffix}/{str(tq)}.model")

    # optNaN.method=2 : Remove leading and closing zeros, optNaN.k=3: See remNaN_spline
    M = InitCond(trans_normalized, S, spec, blocks, optNaN(2, 3))

    # Remove the leading and ending nans:y = X_n for the estimation WITH missing data
    y, indNaN = remNaNs_spline(trans_normalized, optNaN(3, 3))

    # EMstep() and runKF() take y in the form of (var, time)
    M_n, loglik = EMstep(y.T, M, S, blocks, spec)

    pre_loglik = loglik
    LL = np.array([pre_loglik])

    print(f"\nDFM estimation for {str(tq)} with obs. {y.T.columns[0]:%Y-%m} - {y.T.columns[-1]:%Y-%m} with initial loglik {loglik:.2f}")
    # print(f"\nEstimation sample period: {y.columns[0]:%Y-%m-%d (%a)} - {y.columns[-1]:%Y-%m-%d (%a)}")
    # print(f"{nMv} monthly and {nQv} quarterly variables (", *qvs, ")")
    # print(f"Loop started at {last_time:%H:%M:%S}.")

    # number of monthly variable
    nMv = spec.loc[spec.Frequency.eq("m"), "SeriesName"].count()
    # number of Quarterly variable
    nQv = spec.loc[spec.Frequency.eq("q"), "SeriesName"].count()
    qvs = spec.loc[spec.Frequency.eq("q")].index.tolist() # SeriesName -> SeriesID

    # Loop until converges or max iter.
    for num_iter in range(max_iter):

        # DFModel = namedtuple('DFModel', 'A C Q R Z_0 V_0')
        M = DFModel((1 - gamma) * M.A + gamma * M_n.A,
                    (1 - gamma) * M.C + gamma * M_n.C,
                    (1 - gamma) * M.Q + gamma * M_n.Q,
                    (1 - gamma) * M.R + gamma * M_n.R,
                    (1 - gamma) * M.Z_0 + gamma * M_n.Z_0,
                    (1 - gamma) * M.V_0 + gamma * M_n.V_0,)

        # Applying EM algorithm
        M_n, loglik = EMstep(y.T, M, S, blocks, spec)

        if (np.mod(num_iter, 30) == 0):  # Print updates to command window
            new_time = datetime.now()
            print(f"{num_iter:4,.0f}, {new_time:%H:%M:%S}, {(new_time - last_time).seconds:4,.0f} sec, loglik = {loglik:6,.2f},"
                  f"    loglik - pre_loglik = {loglik - pre_loglik:4,.2f}")
            last_time = new_time

        if abs(loglik - pre_loglik) < threshold:
            # converged = 1  # Check convergence
            print(f"Loop ended with loglik {loglik:4,.2f} in {num_iter} iterations")
            break

        pre_loglik = loglik
        LL = np.append(LL, pre_loglik)

    # smoothed factors in the form of (time, state)
    factors = runKF(y.T, M)[0].T.iloc[1:]

    # smoothed (normalized) y in the form of (time, var)
    smoothed_normalized = factors.dot(M.C.T)

    # Unnormalized and smoothed y
    smoothed = smoothed_normalized * std + mean

    if plot_lik:
        fig, axs = plt.subplots(1, 2, figsize=(16, 4))
        Niter = len(LL)
        t1 = int(len(LL) * 9 / 10)
        axs[0].plot(range(Niter), LL)
        axs[1].plot(range(t1, Niter), LL[t1:])

    ER = DFMResult(LL, smoothed, mean, std, smoothed_normalized, factors, M, threshold, max_iter, gamma, y, S, spec, blocks)

    file = open(filename, "wb")
    pickle.dump(ER._asdict(), file)
    file.close()
    print(f"Estimation result is saved as {filename}")

    return ER


def InitCond(X, S, spec, blocks, optnan):

    def bdkg_index(A, A_i, b):
        index = [(bb, ii, jj) for bb, ii, jj in A.index] + [(b, ii, jj) for ii, jj in A_i.index]
        columns = [(bb, ii, jj) for bb, ii, jj in A.columns] + [(b, ii, jj) for ii, jj in A_i.columns]
        A = pd.DataFrame(block_diag(A, A_i), index=index, columns=columns)
        return A

    def bkdg_flat_index(A, B):
        index = A.index.to_flat_index().tolist() + B.index.to_flat_index().tolist()
        columns = A.columns.to_flat_index().tolist() + B.columns.to_flat_index().tolist()
        A = pd.DataFrame(block_diag(A, B), index=index, columns=columns)
        return A

    r, p, Rcon, q = S

    nQ = sum(spec.Frequency.eq("q"))
    # Number of quarterly series
    Qv = spec.loc[spec.Frequency.eq("q")].index.tolist()
    Mv = spec.loc[spec.Frequency.eq("m")].index.tolist()

    pC = Rcon.shape[1]  # 5, 'tent' structure size (quarterly to monthly)
    ppC = max(p, pC)

    # OPTS = pd.Series({'disp':0}) # turns off diagnostic information for eigenvalue computation
    xBal, indNaN = remNaNs_spline(X, optnan)

    T, N = xBal.shape

    xNaN = xBal.copy()
    xNaN[indNaN] = np.nan  # set missing values equal to NaN

    res = xBal.copy()
    resNaN = xNaN.copy()

    # Initialize model coefficient output
    # C = np.empty((25, 0), int)
    C = pd.DataFrame()
    A = pd.DataFrame()
    Q = pd.DataFrame()
    V_0 = pd.DataFrame()

    # Set the first observations as NaNs: For quarterly-monthly aggreg. scheme
    indNaN.iloc[: pC - 1, :] = True

    for r_i, b in zip(r, blocks.columns):  # loop for each block

        # Observation eq.
        C_i = pd.DataFrame(np.zeros((N, r_i * ppC)), index=spec.index)
        # lags = [0, 1, ..., ppC], factors = [1, 2, ..., r_i]
        C_i.columns = pd.MultiIndex.from_product([range(ppC), range(1, r_i + 1)])
        idx_iM = blocks.loc[spec.Frequency.eq("m") & blocks[b].eq(1)].index
        idx_iQ = blocks.loc[spec.Frequency.eq("q") & blocks[b].eq(1)].index

        # Return eigenvectors v with largest r_i eigenvalues d
        d, v = eigh(res.loc[:, idx_iM].cov(), subset_by_index=[idx_iM.shape[0] - r_i, idx_iM.shape[0] - 1],)
        v = pd.DataFrame(v, index=idx_iM, columns=list(range(1, r_i + 1)))

        # Flip sign for cleaner output. Gives equivalent results without this section
        v *= (sum(v.squeeze()) > 0) * 2 - 1

        # For monthly series with loaded blocks (rows), replace with eigenvector
        # This gives the loading
        C_i.loc[idx_iM, 0] = pd.concat([v], axis=1, keys=[0])
        f = res.loc[:, idx_iM].dot(v)
        # Data projection for eigenvector direction
        ####  change???
        F = pd.DataFrame(index=xBal.iloc[ppC-1:].index)

        # Lag matrix using loading. This is later used for quarterly series
        for kk in range(max(p + 1, pC)):
            f_lag = pd.concat([f.shift(kk)], axis=1, keys=[kk])
            F = pd.concat([F, f_lag], axis=1, join="inner")

        Rcon_i = np.kron(Rcon, np.eye(r_i))  # Quarterly-monthly aggregate scheme
        q_i = np.kron(q, np.zeros(r_i))  # Rcon_i * C_i = q_i

        # Produces projected data with lag structure (so pC-1 fewer entries)
        ff = F.iloc[:, :r_i * pC]

        for j in idx_iQ:  # Loop for quarterly variables
            # For series j, values are dropped to accommodate lag structure
            xx_j = resNaN[j].iloc[pC - 1:].copy()

            if (~xx_j.isnull()).sum().squeeze() < ff.shape[1] + 2:
                xx_j = res[j].iloc[pC - 1:]  # Replaces xx_j with spline if too many NaNs

            ff_j = ff.loc[~xx_j.isnull(), :]

            xx_j = xx_j.loc[~xx_j.isnull()]
            iff_j = inv(ff_j.T.dot(ff_j))

            Cc = iff_j.dot(ff_j.T).dot(xx_j)  # least squares

            # Spline data monthly to quarterly conversion
            Cc = Cc - iff_j.dot(Rcon_i.T).dot(inv(Rcon_i.dot(iff_j).dot(Rcon_i.T))).dot(Rcon_i.dot(Cc))
            # Cc = Cc - iff_j.dot(Rcon_i.T).dot(inv(Rcon_i.dot(iff_j).dot(Rcon_i.T))).dot((Rcon_i.dot(Cc)-q_i))

            C_i.loc[j].iloc[: pC * r_i] = Cc

        # Zeros in first pC-1 entries (replace dropped from lag)
        ff = pd.concat([pd.DataFrame(0, index=res.iloc[: pC - 1].index, columns=ff.columns), ff], axis=0,)

        # Residual calculations
        res = res.values - ff.dot(C_i.T)
        resNaN = res.copy()
        resNaN[indNaN] = np.nan

        C = pd.concat([C, pd.concat([C_i], axis=1, keys=[b])], axis=1)

        F.columns = pd.MultiIndex.from_tuples(F.columns)
        F.columns.names = ["p", "f"]

        ## Transition equation
        z = F.loc[:, 0]  # Projected data (no lag)
        Z = F.loc[:, 1:p]  # Data with lag 1

        # Initialize transition matrix
        # f_t = A * f_(t-1)
        A_i = pd.DataFrame(0, index=F.columns, columns=[(i + 1, j) for i, j in F.columns])
        A_i.columns = pd.MultiIndex.from_tuples(A_i.columns)
        A_temp = inv(Z.T.dot(Z)).dot(Z.T).dot(z)  # OLS coefficient of AR(p)

        A_i.loc[idx[0, :], idx[:p, :]] = A_temp.T
        A_i.loc[idx[1:, :], idx[: (ppC - 1), :]] = np.eye(r_i * (ppC - 1))

        Q_i = pd.DataFrame(0, index=F.columns, columns=F.columns)
        e = z.squeeze() - Z.dot(A_temp).squeeze()  # VAR residuals
        Q_i.loc[0, 0] = np.cov(e, rowvar=False)  # VAR covariance matrix

        initV_i = inv(eye((r_i * ppC) ** 2) - kron(A_i, A_i)).dot(Q_i.values.flatten("F")).reshape(r_i * ppC, r_i * ppC, order="F")
        initV_i = pd.DataFrame(initV_i, index=Q_i.index, columns=Q_i.columns)

        # Gives top left block for the transition matrix
        A = bdkg_index(A, A_i, b)
        Q = bdkg_index(Q, Q_i, b)
        V_0 = bdkg_index(V_0, initV_i, b)

    for df in [A, Q, V_0]:
        df.index = pd.MultiIndex.from_tuples(df.index)
        df.columns = pd.MultiIndex.from_tuples(df.columns)

    # C = [C Cm Cy]
    Cm = pd.DataFrame(np.eye(N), index=spec.index, columns=spec.index).loc[:, Mv]
    Cm.columns = pd.MultiIndex.from_product([Cm.columns, [0], [1]])
    C = pd.concat([C, Cm], axis=1)

    Cy = pd.DataFrame(0, index=spec.index, columns=pd.MultiIndex.from_product([Qv, range(pC), [1]]))

    for j in Qv:
        Cy.loc[j, idx[j, :, :]] = [1, 2, 3, 2, 1]

    # Monthly-quarterly aggregate scheme
    C = pd.concat([C, Cy], axis=1)
    R = pd.DataFrame(np.diag(resNaN.var()), index=resNaN.columns, columns=resNaN.columns)

    BM = pd.DataFrame(0, index=Mv, columns=Mv)
    # Initialize monthly transition matrix values
    SM = pd.DataFrame(0, index=Mv, columns=Mv)
    # Initialize monthly residual covariance matrix values

    for v in Mv:
        # Set observation equation residual covariance matrix diagonal
        R.loc[v, v] = 1e-04

        # Subsetting series residuals for series i
        res_i = resNaN.loc[:, v].copy()

        # Returns number of leading/ending zeros
        rem_i = res_i.isnull()
        leadZero = rem_i.cumsum() == list(range(1, rem_i.shape[0] + 1))
        endZero = (rem_i.sum() - rem_i.cumsum() + rem_i) == list(range(rem_i.shape[0], 0, -1))

        # Truncate leading and ending zeros
        res_i = res.loc[:, [v]].copy()
        res_i = res_i.loc[~leadZero & ~endZero]

        # Linear regression: AR 1 process for monthly series residuals
        BM.loc[[v], [v]] = inv(res_i.iloc[:-1].T.dot(res_i.iloc[:-1])).dot(res_i.iloc[:-1].T).dot(res_i.iloc[1:])
        SM.loc[[v], [v]] = (res_i - res_i.shift(1).dot(BM.loc[[v], [v]])).cov()
        # Residual covariance matrix

    sig_e = pd.DataFrame(0, index=Qv, columns=["sig"])

    for v in Qv:
        sig_e.loc[v] = R.loc[v, v] / 19.0
        R.loc[v, v] = 1e-04  # Covariance for obs matrix residuals

    # For BQ, SQ
    rho0 = 0.1
    temp = np.zeros((5, 5))
    temp[0, 0] = 1

    # Blocks for covariance matrices
    SQ = kron(np.diag((1 - rho0 ** 2) * sig_e.squeeze()), temp)
    BQ = kron(np.eye(nQ), np.append([[rho0, 0, 0, 0, 0]], np.append(np.eye(4), np.zeros((4, 1)), axis=1), axis=0))
    initViQ = inv(np.eye((ppC * nQ) ** 2) - kron(BQ, BQ)).dot(SQ.flatten("F")).reshape(ppC * nQ, ppC * nQ)
    initViM = np.diag(1 / np.diag(np.eye(BM.shape[0]) - BM ** 2)) * SM

    BQ = pd.DataFrame(BQ, index=MI.from_product([Qv, range(pC)]), columns=MI.from_product([Qv, range(1, pC + 1)]),)
    SQ = pd.DataFrame(SQ, index=MI.from_product([Qv, range(pC)]), columns=MI.from_product([Qv, range(pC)]),)
    initViQ = pd.DataFrame(initViQ, index=MI.from_product([Qv, range(pC)]), columns=MI.from_product([Qv, range(pC)]),)

    # Output
    BM.index = BM.columns = Cm.columns
    BQ.index = BQ.columns = Cy.columns
    SM.index = SM.columns = Cm.columns
    SQ.index = SQ.columns = Cy.columns
    initViM.index = initViM.columns = Cm.columns
    initViQ.index = initViQ.columns = Cy.columns

    A = bkdg_flat_index(bkdg_flat_index(A, BM), BQ)  # Observation matrix
    Q = bkdg_flat_index(bkdg_flat_index(Q, SM), SQ)  # Residual covariance matrix (transition)
    Z_0 = pd.Series(np.zeros(A.shape[0]), index=A.index)  # States
    V_0 = bkdg_flat_index(bkdg_flat_index(V_0, initViM), initViQ)  # Covariance of states

    A.index = A.columns = MI.from_tuples(A.index)
    Q.index = Q.columns = MI.from_tuples(Q.index)
    V_0.index = V_0.columns = pd.MultiIndex.from_tuples(Q.index)
    Z_0.index = pd.MultiIndex.from_tuples(Q.index)

    return DFModel(A, C, Q, R, Z_0, V_0)


def remNaNs_spline(X_0, optNaN):

    X = X_0.copy()
    indNaN = X.isnull()
    T, N = X.shape

    def nanLE(rem1, X):
        nanLead = rem1.cumsum() == list(range(1, rem1.shape[0] + 1))
        nanEnd = (rem1.sum() - rem1.cumsum() + rem1) == list(range(rem1.shape[0], 0, -1))
        nanLE = nanLead | nanEnd
        X = X.loc[~nanLE]
        indNaN = X.isnull()
        return X, indNaN

    def filtering(X, k):
        Y = X.copy()
        for var in X.columns:
            x = X[var].copy()
            isnanx = x.isnull()
            t1 = isnanx[isnanx.eq(False)].index[0]
            t2 = isnanx[isnanx.eq(False)].index[-1]
            x[t1:t2] = x[t1:t2].interpolate("cubic")
            isnanx = x.isnull()
            x.loc[x.isnull()] = x.median(skipna=True)
            x_ext = np.append(np.append([x.values[0]] * k, x.values), [x.values[-1]] * k)
            x_ma = lfilter(np.ones(2 * k + 1) / (2 * k + 1), 1, x_ext)
            x_ma = x_ma[2 * k :]
            x_repl = pd.Series(x_ma, index=x.index)
            x[isnanx] = x_repl[isnanx]
            Y.loc[:, var] = x
        return Y

    # replace all the missing values
    if optNaN.method == 1:
        for var in X.columns:
            x = X[var].copy()
            isnanx = x.isnull()
            x.loc[isnanx] = x.median(skipna=True)
            x_ext = np.append(
                np.append([x.values[0]] * optNaN.k, x.values), [x.values[-1]] * optNaN.k
            )
            x_ma = lfilter(np.ones(2 * optNaN.k + 1) / (2 * optNaN.k + 1), 1, x_ext)
            x_ma = x_ma[2 * optNaN.k :]
            x_repl = pd.Series(x_ma, index=x.index)
            x[isnanx] = x_repl[isnanx]
            X[var] = x

    # replace missing values after removing leading and closing zeros
    elif optNaN.method == 2:
        rem1 = indNaN.sum(axis=1) > 0.8 * N
        X, indNaN = nanLE(rem1, X)
        X = filtering(X, optNaN.k)

    # only remove rows with leading and closing zeros
    elif optNaN.method == 3:
        rem1 = indNaN.sum(axis=1) == N
        X, indNaN = nanLE(rem1, X)

    # remove rows with leading and closing zeros & replace missing values
    elif optNaN.method == 4:
        rem1 = indNaN.sum(axis=1) == N
        X, indNaN = nanLE(rem1, X)
        X = filtering(X, optNaN.k)

    # replace missing values
    elif optNaN.method == 5:
        indNaN = X.isnull()
        X = filtering(X, optNaN.k)

    return X, indNaN


def EMstep(y, M, S, blocks, spec):
    """
    Applies EM algorithm for parameter reestimation
    #
    Description:
        EMstep reestimates parameters based on the Estimation Maximization (EM) algorithm.
        This is a two-step procedure:
            (1) E-step: the expectation of the log-likelihood is calculated using previous parameter estimates.
            (2) M-step: Parameters are re-estimated through the maximisation of the log-likelihood
                (maximize result from (1)).

    See "Maximum likelihood estimation of factor models on data sets with arbitrary pattern of missing data"
    for details about parameter derivation (Banbura & Modugno, 2010). This procedure is in much the same spirit.

    Input:
        M     : DFM parameters (A, C, Q, R, Z_0, V_0)
        S     : DFM structure (r, p, Rcon, q)
        blocks: Block structure for each series (i.e. for a series, the structure
                [1 0 0 1] indicates loadings on the first and fourth factors)
        spec  : spec for observed variables such as name, frequency, transformation, etc.

    Output:
        C_new  : Updated observation matrix
        R_new  : Updated covariance matrix for residuals of observation matrix
        A_new  : Updated transition matrix
        Q_new  : Updated covariance matrix for residuals for transition matrix
        Z_0    : Initial value of state
        V_0    : Initial value of covariance matrix
        loglik : Log likelihood

    References:
        "Maximum likelihood estimation of factor models on data sets with arbitrary pattern of
        missing data" by Banbura & Modugno (2010). Abbreviated as BM2010
    """

    ## Initialize preliminary values
    # Store series/model values

    r, p, Rcon, q = S

    A = M.A.copy()
    C = M.C.copy()
    Q = M.Q.copy()
    R = M.R.copy()
    Z_0 = M.Z_0.copy()
    V_0 = M.V_0.copy()

    Qv = spec.loc[spec.Frequency.eq("q")].index.tolist()
    Mv = spec.loc[spec.Frequency.eq("m")].index.tolist()

    n, T = y.shape
    nQ = len(Qv)
    nM = n - nQ
    # Number of monthly series
    pC = Rcon.shape[1]
    ppC = max(p, pC)
    num_blocks = blocks.shape[1]  # Number of blocks

    # Set missing data series values to 0
    y0 = y.copy()
    nanY = y0.isnull()
    y0[nanY] = 0

    # Initialize output
    A_new = A.copy()
    C_new = C.copy()
    Q_new = Q.copy()
    V_0_new = V_0.copy()

    ## ESTIMATION STEP: Compute the (expected) sufficient statistics for a single
    # Kalman filter sequence

    # Running the Kalman filter and smoother with current parameters
    # Note that log-liklihood is NOT re-estimated after the runKF step: This
    # effectively gives the previous iteration's log-likelihood
    # For more information on output, see runKF
    Zsmooth, Vsmooth, VVsmooth, loglik, ps, ps_0 = runKF(y, M)

    ## MAXIMIZATION STEP (TRANSITION EQUATION)
    # See (Banbura & Modugno, 2010) for details.

    ### 2A. UPDATE FACTOR PARAMETERS INDIVIDUALLY ----------------------------
    for b in blocks.columns:  # Loop for each block: factors are uncorrelated

        # ESTIMATE FACTOR PORTION OF Q, A
        # Note: EZZ, EZZ_BB, EZZ_FB are parts of equations 6 and 8 in BM 2010

        # E[f_t*f_t' | Omega_T]
        EZZ = (Zsmooth.loc[b, ps_0[1:]].dot(Zsmooth.loc[b, ps_0[1:]].T)
               + Vsmooth.loc[b, idx[ps_0[1:], b]].groupby(level=[1, 2, 3], axis=1).sum()[b])

        # E[f_{t-1}*f_{t-1}' | Omega_T]
        EZZ_BB = (Zsmooth.loc[b, ps_0[:-1]].dot(Zsmooth.loc[b, ps_0[:-1]].T)
                  + Vsmooth.loc[b, idx[ps_0[:-1], b]].groupby(level=[1, 2, 3], axis=1).sum()[b])

        # E[f_t*f_{t-1}' | Omega_T]
        EZZ_FB = (Zsmooth.loc[b, ps_0[1:]].dot(Zsmooth.shift(axis=1).loc[b, ps_0[1:]].T)
                  + VVsmooth.loc[b, idx[:, b]].groupby(level=[1, 2, 3], axis=1).sum()[b])

        # Select transition matrix/covariance matrix for block i
        A_i = A.loc[b, b].copy()
        Q_i = Q.loc[b, b].copy()

        # Equation 6: Estimate VAR(p) for factor
        A_i.loc[0, :p-1] = EZZ_FB.loc[0, :p-1].dot(inv(EZZ_BB.loc[idx[:p-1], idx[:p-1]])).copy().values

        # Equation 8: Covariance matrix of residuals of VAR
        Q_i.loc[0, 0] = (EZZ.loc[0, 0] - A_i.loc[0, idx[: p - 1]].dot(EZZ_FB.loc[0, idx[: p - 1]].T)).copy().values / T

        # Place updated results in output matrix
        A_new.loc[idx[b, :, :], idx[b, :, :]] = A_i.copy().values
        Q_new.loc[idx[b, :, :], idx[b, :, :]] = Q_i.copy().values
        V_0_new.loc[idx[b, :, :], idx[b, :, :]] = Vsmooth.loc[b, idx[ps_0[0], b]].copy().values

    ### 2B. UPDATING PARAMETERS FOR IDIOSYNCRATIC COMPONENT ------------------

    # Below 3 estimate the idiosyncratic component (for eqns 6, 8 BM 2010)
    # E[f_t*f_t' | \Omega_T]
    EZZ = (diag(diag(Zsmooth.loc[spec.index, ps_0[1:]].dot(Zsmooth.loc[spec.index, ps_0[1:]].T)))
           + diag(diag(Vsmooth.loc[spec.index, idx[ps_0[1:], spec.index]].groupby(level=[1, 2, 3], axis=1, sort=False).sum())))

    # E[f_{t-1}*f_{t-1}' | \Omega_T]
    EZZ_BB = (diag(diag(Zsmooth.loc[spec.index, ps_0[:-1]].dot(Zsmooth.loc[spec.index, ps_0[:-1]].T)))
              + diag(diag(Vsmooth.loc[spec.index, idx[ps_0[:-1], spec.index]].groupby(level=[1, 2, 3], axis=1, sort=False).sum())))

    # E[f_t*f_{t-1}' | \Omega_T]
    EZZ_FB = (diag(diag(Zsmooth.loc[spec.index, ps_0[1:]].dot(Zsmooth.shift(axis=1).loc[spec.index, ps_0[1:]].T)))
              + diag(diag(VVsmooth.loc[spec.index, idx[:, spec.index]].groupby(level=[1, 2, 3], axis=1, sort=False).sum())))

    states_MQv = Zsmooth.loc[spec.index, ps_0[1:]].index
    A_i = pd.DataFrame(EZZ_FB.dot(diag(1 / diag(EZZ_BB))), index=states_MQv, columns=states_MQv).copy()  # Equation 6
    Q_i = pd.DataFrame((EZZ - A_i.values.dot(EZZ_FB.T)) / T, index=states_MQv, columns=states_MQv).copy()  # Equation 8

    # Place updated results in output matrix
    A_new.loc[idx[Mv, :, :], idx[Mv, :, :]] = A_i.loc[Mv, Mv].copy().values
    Q_new.loc[idx[Mv, :, :], idx[Mv, :, :]] = Q_i.loc[Mv, Mv].copy().values
    V_0_new.loc[idx[Mv, :, :], idx[Mv, :, :]] = diag(diag(Vsmooth.copy().loc[Mv, idx[ps_0[0], Mv]]))

    ## 3 MAXIMIZATION STEP (observation equation)

    ### INITIALIZATION AND SETUP ----------------------------------------------
    Z_0_new = Zsmooth.loc[:, ps_0[0]].copy()  # zeros(size(Zsmooth,1),1); #

    R_con = empty((0, 0), int)
    q_con = empty((0, 1), int)

    for r_i in r:
        R_con = block_diag(R_con, kron(Rcon, eye(r_i)))
        q_con = np.append(q_con, np.zeros((r_i * Rcon.shape[0], 1)), axis=0)

    R_con = pd.DataFrame(R_con, columns=A.loc[blocks.columns, blocks.columns].columns)

    blocks_sum = blocks.copy()
    blocks_sum["bc"] = blocks.dot([1, 2, 3, 4])
    blocks_sum = (blocks_sum.set_index("bc", append=True).reorder_levels([1, 0], axis=0).sort_index(axis=0))
    bl = blocks_sum.drop_duplicates().loc[[1, 5, 4, 3]]

    for i, row in bl.eq(1).iloc[:3].iterrows():

        bs = blocks_sum.drop_duplicates().columns[row].tolist()
        rs = sum(r[bs])

        idx_iM = [v for v in Mv if v in blocks_sum.loc[i[0]].index.tolist()]  # variables need to be ordered as in Mv!
        n_i = len(idx_iM)

        # Initialize sums in equation 13 of BGR 2010
        denom = zeros((n_i * rs, n_i * rs))
        nom = zeros((n_i, rs))

        ### UPDATE MONTHLY VARIABLES: Loop through each period ----------------
        Wd = ~nanY.loc[idx_iM, :].to_numpy()
        Zd = Zsmooth.loc[idx[bs, 0, :], :].to_numpy()
        Vd = Vsmooth.loc[idx[bs, 0, :], idx[:, bs, 0, :]].to_numpy().T.reshape(T + 1, rs, rs)
        yd = y0.loc[idx_iM, :].to_numpy()
        Ze = Zsmooth.loc[idx[idx_iM, 0, :], :].to_numpy()
        Ve = Vsmooth.loc[idx[bs, 0, :], idx[:, idx_iM, 0, :]].to_numpy().T.reshape(T + 1, n_i, rs)

        for t in range(y.shape[1]):
            Wt = diag(Wd[:, t])  # Gives selection matrix (1 for nonmissing values)

            # E[f_t*t_t' | Omega_T]
            denom = denom + kron(Zd[:, [t + 1]].dot(Zd[:, [t + 1]].T) + Vd[t + 1, :, :], Wt)

            # E[y_t*f_t' | \Omega_T]
            nom = nom + yd[:, [t]].dot(Zd[:, [t + 1]].T) - Wt[:, :].dot(Ze[:, [t + 1]].dot(Zd[:, [t + 1]].T) + Ve[t + 1, :, :])

        vec_C = inv(denom).dot(nom.flatten("F"))  # Eqn 13 BGR 2010

        # Place updated monthly results in output matrix
        C_new.loc[idx_iM, idx[bs, 0, :]] = vec_C.reshape(n_i, rs, order="F")

        ### UPDATE QUARTERLY VARIABLES -----------------------------------------
        idx_iQ = [v for v in Qv if v in blocks_sum.loc[i[0]].index.tolist()]  # Index for quarterly series

        # Monthly-quarterly aggregation scheme
        R_con_i = R_con.loc[:, bs].to_numpy()
        q_con_i = q_con.copy()

        # select non-zero rows of R_con_i, q_con_i
        sel_rows = np.any(R_con_i, axis=1)
        R_con_i = R_con_i[sel_rows, :]
        q_con_i = q_con_i[sel_rows, :]

        # print('\n i row:', i, '\n', row, '\n bs', bs, '\n idx_iQ', idx_iQ, '\n')
        # Loop through quarterly series in loading. This parallels monthly code
        for j in idx_iQ:
            # print('loop thru: ', j)

            # Initialization
            denom = zeros((rs * ppC, rs * ppC))
            nom = zeros((1, rs * ppC))

            # Place quarterly values in output matrix
            V_0_new.loc[idx[j, :, :], idx[j, :, :]] = Vsmooth[ps_0[0]].loc[idx[j, :, :], idx[j, :, :]].copy()
            A_new.loc[idx[j, 0, 1], idx[j, 0, 1]] = A_i.loc[idx[j, 0, 1], idx[j, 0, 1]].copy()
            Q_new.loc[idx[j, 0, 1], idx[j, 0, 1]] = Q_i.loc[idx[j, 0, 1], idx[j, 0, 1]].copy()

            Wd = ~nanY.loc[[j], :].to_numpy()
            Zd = Zsmooth.loc[idx[bs, :, :], :].to_numpy()
            Vd = Vsmooth.loc[idx[bs, :, :], idx[:, bs, :, :]].to_numpy().T.reshape(T + 1, rs * ppC, rs * ppC)
            yd = y0.loc[[j], :].to_numpy()
            Ze = Zsmooth.loc[idx[j, :, :], :].to_numpy()
            Ve = Vsmooth.loc[idx[bs, :, :], idx[:, j, :, :]].to_numpy().T.reshape(T + 1, ppC, rs * ppC)

            for t in range(y.shape[1]):
                Wt = diag(Wd[:, t])  # Selection matrix for quarterly values

                # Intermediate steps in BGR equation 13
                denom = denom + kron(Zd[:, [t + 1]].dot(Zd[:, [t + 1]].T) + Vd[t + 1, :, :], Wt)
                nom = nom + yd[:, [t]].dot(Zd[:, [t + 1]].T)
                nom = nom - Wt.dot(np.array([[1, 2, 3, 2, 1]]).dot(Ze[:, [t + 1]]).dot(Zd[:, [t + 1]].T)
                                   + np.array([[1, 2, 3, 2, 1]]).dot(Ve[t + 1, :, :]))

            # print(denom.shape, nom.shape)
            C_i = inv(denom).dot(nom.T)

            # BGR equation 13
            inv_RdR = inv(R_con_i.dot(inv(denom)).dot(R_con_i.T))
            C_i_constr = C_i - inv(denom).dot(R_con_i.T).dot(inv_RdR).dot(R_con_i.dot(C_i) - q_con_i)

            # Place updated values in output structure
            C_new.loc[[j], idx[bs, :, :]] = C_i_constr.T

    ### 3B. UPDATE COVARIANCE OF RESIDUALS FOR OBSERVATION EQUATION -----------
    # Initialize covariance of residuals of observation equation
    R_new = zeros((n, n))

    Wd = ~nanY.to_numpy()
    Zd = Zsmooth.to_numpy()
    Vd = Vsmooth.to_numpy().T.reshape(T + 1, A.shape[0], A.shape[1])
    yd = y0.loc[:, :].to_numpy()
    Ze = Zsmooth.loc[idx[j, :, :], :].to_numpy()
    Ve = Vsmooth.loc[idx[bs, :, :], idx[:, j, :, :]].to_numpy().T.reshape(T + 1, ppC, rs * ppC)

    for t in range(y.shape[1]):
        Wt = diag(Wd[:, t])  # Selection matrix
        # BGR equation 15
        R_new = (R_new + (yd[:, [t]] - Wt.dot(C_new).dot(Zd[:, [t + 1]])).dot((yd[:, [t]] - Wt.dot(C_new).dot(Zd[:, [t + 1]])).T)
                 + Wt.dot(C_new).dot(Vd[t + 1, :, :]).dot(C_new.T).dot(Wt)
                 + (eye(n) - Wt).dot(R).dot(eye(n) - Wt))

    R_new = R_new / T
    R_new = pd.DataFrame(diag(diag(R_new)), index=R.index, columns=R.columns)  # RR(RR<1e-2) = 1e-2;

    for v in Mv:
        R_new.loc[v, v] = 1e-04

    for v in Qv:
        R_new.loc[v, v] = 1e-04

    # in FRBNY matlab code
    # V_0 is returen instead of V_0_new AND
    # Z_0_new is not defined, but Z_0's value is updated and returned!

    return DFModel(A_new, C_new, Q_new, R_new, Z_0_new, V_0), loglik


def dfm_copy(M, to_numpy=True):
    """Input: DFModel M, to_numpy True/False
    Output: 6 np.ndarrays A, C, Q, R, Z_0, V_0

    From M, 6 dataframes A, C, Q, R, Z_0, V_0 are copied.
    By default (to_numpy=True) the parameters are converted to np.ndarrays."""

    A, C, Q, R, Z_0, V_0 = M

    if to_numpy:
        return (A.to_numpy().copy(),
                C.to_numpy().copy(),
                Q.to_numpy().copy(),
                R.to_numpy().copy(),
                Z_0.to_numpy().copy(),
                V_0.to_numpy().copy(),)
    else:
        return A.copy(), C.copy(), Q.copy(), R.copy(), Z_0.copy(), V_0.copy()


def runKF(y, M):
    """
    runKF() applies a Kalman filter and fixed-interval smoother. The script uses the following model:
        Y_t = C Z_t + e_t for e_t ~ N(0, R)
        Z_t = A Z_{t-1} + mu_t for mu_t ~ N(0, Q)

    Input
        Y   : k-by-nobs matrix of input data
        A   : m-by-m transition matrix
        C   : k-by-m observation matrix
        Q   : m-by-m covariance matrix for transition equation residuals (mu_t)
        R   : k-by-k covariance for observation matrix residuals (e_t)
        Z_0 : 1-by-m vector, initial value of state
        V_0 : m-by-m matrix, initial value of state covariance matrix

    Output
        zsmooth  : m-by-(nobs+1) matrix, smoothed factor estimates (i.e. zsmooth(:,t+1) = Z_t|T)
        Vsmooth  : m-by-m-by-(nobs+1) array, smoothed factor covariance matrices (i.e. Vsmooth(:,:,t+1) = Cov(Z_t|T))
        VVsmooth : m-by-m-by-nobs array, lag 1 factor covariance matrices (i.e. Cov(Z_t,Z_t-1|T))
        loglik   : scalar, log-likelihood
        ps       : dates 1, 2, 3, 4, ..., T
        ps_0     : dates 0, 1, 2, 3, 4, ..., T

    References:
        QuantEcon's "A First Look at the Kalman Filter"
        Adapted from replication files for:
        "Nowcasting", 2010, (by Marta Banbura, Domenico Giannone and Lucrezia Reichlin),
        in Michael P. Clements and David F. Hendry, editors, Oxford Handbook on Economic Forecasting.

    The software can be freely used in applications. Users are kindly requested to add acknowledgements to published work and
    to cite the above reference in any resulting publications.
    """

    states = MI.from_tuples(M.A.index)
    ps = y.columns
    ps_0 = pd.date_range(ps[0] - MonthEnd(), ps[-1], freq=ps.freq)

    ps_states = MI.from_tuples([(i, j, p, q) for i in ps for j, p, q in states.to_flat_index()])
    ps_0_states = MI.from_tuples([(i, j, p, q) for i in ps_0 for j, p, q in states.to_flat_index()])

    Y = y.to_numpy().copy()
    A, C, Q, R, Z_0, V_0 = dfm_copy(M)

    Zm, ZmU, Vm, VmU, loglik, k_t = SKF(Y, A, C, Q, R, Z_0, V_0)  # Kalman filter
    ZmT, VmT, VmT_1 = FIS(A, Zm, ZmU, Vm, VmU, loglik, k_t)  # Fixed interval smoother

    # Organize output
    Zsmooth = pd.DataFrame(ZmT.T, index=states, columns=ps_0)

    VmT = np.moveaxis(VmT, 0, -1).reshape(len(states), len(ps_0_states), order="F")
    VmT_1 = np.moveaxis(VmT_1, 0, -1).reshape(len(states), len(ps_states), order="F")

    Vsmooth = pd.DataFrame(VmT, index=states, columns=ps_0_states)
    VVsmooth = pd.DataFrame(VmT_1, index=states, columns=ps_states)

    return Zsmooth, Vsmooth, VVsmooth, loglik, ps, ps_0


def SKF(Y, A, C, Q, R, Z_0, V_0):
    """
    SKF() applies the Kalman filter

    Input parameters:
        Y    : k-by-nobs matrix of input data
        A    : m-by-m transition matrix
        C    : k-by-m observation matrix
        Q    : m-by-m covariance matrix for transition equation residuals (mu_t)
        R    : k-by-k covariance for observation matrix residuals (e_t)
        Z_0  : 1-by-m vector, initial value of state
        V_0  : m-by-m matrix, initial value of state covariance matrix

    Output parameters:
        Zm     : m-by-nobs matrix, prior/predicted factor state vector, Zm(:,t) = Z_t|t-1
        ZmU    : m-by-(nobs+1) matrix, posterior/updated state vector, Zm(t+1) = Z_t|t
        Vm     : m-by-m-by-nobs array, prior/predicted covariance of factor state vector, Vm(:,:,t) = V_t|t-1
        VmU    : m-by-m-by-(nobs+1) array, posterior/updated covariance of factor state vector, VmU(:,:,t+1) = V_t|t
        loglik : scalar, value of likelihood function
        k_t    : k-by-m Kalman gain
    """

    def MissData(y, C, R):
        # Returns 1 for nonmissing series
        ix = ~np.isnan(y)
        # Index for columns with nonmissing variables
        e = np.eye(y.shape[0])
        L = e[:, ix]
        # Removes missing series
        y = y[ix]
        # Removes missing series from observation matrix
        C = C[ix, :]
        # Removes missing series from transition matrix
        R = R[ix, ix]
        return y, C, R, L

    ## INITIALIZE OUTPUT VALUES ---------------------------------------------
    # Output structure & dimensions of state space matrix
    k, m = C.shape  # k * m: observable variables and factors

    # Outputs time for data matrix. "number of observations"
    nobs = Y.shape[1]

    # Instantiate output
    Zm = np.ones((nobs, m)) * np.nan  # Z_t | t-1 (prior)
    Vm = np.ones((nobs, m, m)) * np.nan  # V_t | t-1 (prior)
    ZmU = np.ones((nobs + 1, m)) * np.nan
    # Z_t | t (posterior/updated)
    VmU = np.ones((nobs + 1, m, m)) * np.nan
    # V_t | t (posterior/updated)
    loglik = 0

    ## SET INITIAL VALUES ----------------------------------------------------
    Zu = Z_0.copy()  # Z_0|0 (In below loop, Zu gives Z_t | t)
    Vu = V_0.copy()  # V_0|0 (In below loop, Vu guvse V_t | t)

    # Store initial values
    ZmU[0, :] = Zu.copy()
    VmU[0, :, :] = Vu.copy()

    ## KALMAN FILTER PROCEDURE ----------------------------------------------
    for t in range(nobs):

        ### CALCULATING PRIOR DISTIBUTION----------------------------------
        # Use transition eqn to create prior estimate for factor
        # i.e. Z = Z_t|t-1
        Z = A.dot(Zu)

        # Prior covariance matrix of Z (i.e. V = V_t|t-1)
        # Var(Z) = Var(A*Z+u_t) = A*Vu*A' + Q
        V = A.dot(Vu).dot(A.T) + Q
        V = (V + V.T) / 2  # Trick to make symmetric

        ### CALCULATING POSTERIOR DISTRIBUTION ----------------------------
        # Removes missing series: These are removed from Y, C, and R
        Y_t, C_t, R_t, L = MissData(Y[:, t], C, R)

        # Check if y_t contains no data. If so, replace Zu and Vu with prior.
        if Y_t.size == 0:
            Zu = Z
            Vu = V

        else:
            # Steps for variance and population regression coefficients:
            # Var(c_t*Z_t + e_t) = c_t Var(Z_t) c_t' + Var(u) = c_t*V *c_t' + R
            VC = V.dot(C_t.T)
            iF = inv(C_t.dot(VC) + R_t)

            # Matrix of population regression coefficients (QuantEcon eqn #4)
            VCF = VC.dot(iF)

            # Gives difference between actual and predicted observation
            # matrix values
            innov = Y_t - C_t.dot(Z)

            # Update estimate of factor values (posterior)
            Zu = Z + VCF.dot(innov)

            # Update covariance matrix (posterior) for time t
            Vu = V - VCF.dot(VC.T)
            Vu = (Vu + Vu.T) / 2  # Approximation trick to make symmetric

            # Update log likelihood
            loglik = loglik + 0.5 * (log(det(iF)) - innov.T.dot(iF).dot(innov))

        ### STORE OUTPUT----------------------------------------------------
        # Store covariance and observation values for t-1 (priors)
        Zm[t, :] = Z
        Vm[t, :, :] = V

        # Store covariance and state values for t (posteriors)
        # i.e. Zu = Z_t|t   & Vu = V_t|t
        ZmU[t + 1, :] = Zu
        VmU[t + 1, :, :] = Vu

    # Store Kalman gain k_t
    if Y_t.size == 0:
        k_t = np.zeros((m, m))
    else:
        k_t = VCF.dot(C_t)

    return Zm, ZmU, Vm, VmU, loglik, k_t


def FIS(A, Zm, ZmU, Vm, VmU, loglik, k_t):
    """
    Applies fixed-interval smoother

    Description:
        FIS() applies a fixed-interval smoother, and is used in conjunction with SKF().
        See  page 154 of 'Forecasting, structural time series models and the Kalman filter'
        for more details (Harvey, 1990).

    Input parameters:
        A     : m-by-m transition matrix
        Zm    : m-by-nobs matrix, prior/predicted factor state vector, Zm(:,t) = Z_t|t-1
        ZmU   : m-by-(nobs+1) matrix, posterior/updated state vector, Zm(t+1) = Z_t|t
        Vm    : m-by-m-by-nobs array, prior/predicted covariance of factor state vector, Vm(:,:,t) = V_t|t-1
        VmU   : m-by-m-by-(nobs+1) array, posterior/updated covariance of factor state vector, VmU(:,:,t+1) = V_t|t
        loglik: scalar, value of likelihood function
        k_t   : k-by-m Kalman gain

    Output parameters:
        ZmT   : m-by-(nobs+1) matrix, smoothed states, ZmT(:,t+1) = Z_t|T
        VmT   : m-by-m-by-(nobs+1) array, smoothed factor covariance matrices VmT(:,:,t+1) = V_t|T = Cov(Z_t|T)
        VmT_1 : m-by-m-by-nobs array, smoothed lag 1 factor covariance matrices VmT_1(:,:,t) = Cov(Z_t Z_t-1|T)

    Model:
        Y_t = C Z_t + e_t for e_t ~ N(0, R)
        Z_t = A Z_{t-1} + mu_t for mu_t ~ N(0, Q)"""

    ## ORGANIZE INPUT ---------------------------------------------------------
    # Initialize output matrices
    nobs, m = Zm.shape
    ZmT = np.zeros((nobs + 1, m))
    VmT = np.zeros((nobs + 1, m, m))

    # Fill the final period of ZmT, VmT with SKF() posterior values
    ZmT[nobs, :] = ZmU[nobs, :].squeeze()
    VmT[nobs, :, :] = VmU[nobs, :, :].squeeze()

    # Initialize VmT_1 lag 1 covariance matrix for final period
    VmT_1 = np.zeros((nobs, m, m))
    VmT_1[-1, :, :] = (np.eye(m) - k_t).dot(A).dot(VmU[nobs - 1, :, :].squeeze())

    # Used for recursion process. See companion file for details
    J_2 = VmU[nobs - 1, :, :].squeeze().dot(A.T).dot(pinv(Vm[nobs - 1, :, :]))

    ## RUN SMOOTHING ALGORITHM ----------------------------------------------
    # Loop through time reverse-chronologically (starting at final period nobs)
    for t in range(nobs - 1, -1, -1):
        # Store posterior and prior factor covariance values
        VmUt = VmU[t, :, :].squeeze()
        Vmt = Vm[t, :, :].squeeze()

        # Store previous period smoothed factor covariance and lag-1 covariance
        V_T = VmT[t + 1, :, :].squeeze()
        V_T1 = VmT_1[t, :, :].squeeze()

        J_1 = J_2.copy()

        # Update smoothed factor estimate
        ZmT[t, :] = ZmU[t, :] + J_1.dot(ZmT[t + 1, :] - A.dot(ZmU[t, :]))

        # Update smoothed factor covariance matrix
        VmT[t, :, :] = VmUt + J_1.dot(V_T - Vmt).dot(J_1.T)

        if t > 0:
            # Update weight
            J_2 = VmU[t - 1, :, :].squeeze().dot(A.T).dot(pinv(Vm[t - 1, :, :].squeeze()))

            # Update lag 1 factor covariance matrix
            VmT_1[t - 1, :, :] = VmUt.dot(J_2.T) + J_1.dot(V_T1 - A.dot(VmUt)).dot(J_2.T)

    return ZmT, VmT, VmT_1

# nowcasting

In [ ]:
def weekly_nowcasting_fast(tq, vintages, specfile, dfm_model, data, suffix=''):

    print(f"{str(tq)} nowcasting ", end=" ")

    dfm_rtf_fast = pd.DataFrame(np.nan, index=vintages, columns=['dfm'])
    RTF = (f"./rtf/{str(tq)}")

    if not(os.path.exists(RTF)):
        os.makedirs(RTF)

    for week in vintages:
        transformed, mean, std, normalized, raw = load_data(f"{data}/{week}.xlsx", specfile, sample_start)
        fps = pd.date_range(transformed.index[0], transformed.index[-1] + MonthEnd(12), freq=transformed.index.freq)
        transformed = transformed.reindex(fps)
        predicted = predict(transformed, dfm_model)

        predicted = transformed.fillna(predicted).loc[:tq.strftime('%Y-%m-%d')]
        predicted.index.name = "Time"

        if math.isnan(raw.reindex(fps).loc[tq.strftime('%Y-%m-%d'), 'N_gdp']):
            predicted.to_csv(f"{RTF}/{week}{suffix}_predicted.csv")  # SAVE DFMest data to be shared with ML
            dfm_rtf_fast.loc[week, 'dfm'] = predicted.loc[tq.strftime('%Y-%m-%d'), 'N_gdp']
        else:
            dfm_rtf_fast.loc[week, 'dfm'] = dfm_rtf_fast.shift(1).loc[week, 'dfm']

        print('.', end="")

    dfm_rtf_fast.index = pd.to_datetime(dfm_rtf_fast.index)
    dfm_rtf_fast.to_pickle(f'{RTF}/dfm{suffix}_predicted.pkl')
    print(f" saved as {RTF}/dfm{suffix}_rtf.pkl")

    return dfm_rtf_fast


def weekly_nowcasting(tq, vintages, specfile, ER, data, suffix='', print_detail=False):
    """run update_nowcasting for each vintage in vintages

    nowcasts = weekly_nowcasting(vintages, ER, target, print_detail=False)

    Input
        tq           : target quarter
        vintages     : list of dates in which nowcasting is performed
        specfile     : the path to the file that contains DFM model setup, variable transformation methods, etc.
        ER           : DFM result
        RTF          : the path to the folder that contains RTF files
        redo         : similar to what is defined in estimate()
        override     : similar to what is defined in estimate()
        print_detail : if TRue, display detailed update in RTF

    Output
        dfm_rtf      : a dataframe with vintages in index and nowcasting estimate and impacts by each category in columns
    """
    #####################################################################################
    # This part is to return the previously calculated RTF.
    # If there is no previously calculated RTF or redo is True, the MAIN part below starts
    #####################################################################################
    """
    if not(os.path.exists(f'{RTF}/dfm_rtf.xlsx')) or redo:
        pass
    else:
        dfm_rtf = pd.read_excel(f"{RTF}/dfm_rtf.xlsx", sheet_name="data", index_col=0)
        print(f"\nDFM RTF is read from {RTF}.")
        return dfm_rtf
    """

    #####################################################################################
    # MAIN
    #####################################################################################
    RTF = (f"./rtf/{str(tq)}")

    target = Target("N_gdp", tq.strftime('%Y-%m-%d'))

    if not(os.path.exists(RTF)):
        os.makedirs(RTF)

    transformed, mean, std, normalized, raw = load_data(f"{data}/{vintages[0]}.xlsx", specfile, sample_start)
    #X_new, _ = load_data(path.join("input", "mdata", vintages[0] + ".xlsx"), specfile)
    fps = pd.date_range(transformed.index[0], transformed.index[-1] + MonthEnd(12), freq=transformed.index.freq)
    transformed = transformed.reindex(fps)

    predicted = predict(transformed, ER)
    df_n = transformed.fillna(predicted).loc[:tq.strftime('%Y-%m-%d')]
    df_n.index.name = "Time"

    # SAVE DFMest data to be shared with ML
    df_n.to_csv(f"{RTF}/{vintages[0]}{suffix}_predicted.csv")

    nowcasts = pd.DataFrame(index=vintages, columns=["old", "rev", "news", "new"])
    nowcastv = pd.DataFrame()

    nowcasts.loc[vintages[0], "old"] = np.nan
    nowcasts.loc[vintages[0], "rev"] = np.nan
    nowcasts.loc[vintages[0], "news"] = np.nan
    nowcasts.loc[vintages[0], "new"] = predicted.loc[tq.strftime('%Y-%m-%d'), 'N_gdp']

    print("\nDFM RTF is running...", end=" ")

    for v_new, v_old in zip(vintages[1:], vintages[:-1]):
        transformed, mean, std, normalized, raw = load_data(f"{data}/{v_new}.xlsx", specfile, sample_start)
        if not(math.isnan(raw.reindex(fps).loc[tq.strftime('%Y-%m-%d'), 'N_gdp'])):
            break

        try:
            old_cast, rev, news, new_cast, table, DFMest = update_nowcast(
                ER, target, v_old, v_new, specfile, data, print_detail=print_detail)

            # set filename to which estimation result is saved
            DFMest = DFMest.truncate(after=target.t)
            DFMest.index.name = "Time"

            # SAVE DFMest data to be shared with ML
            DFMest.to_csv(f"{RTF}/{v_new}_rtf{suffix}.csv")

            nowcasts.loc[v_new, "old"] = old_cast
            nowcasts.loc[v_new, "rev"] = rev
            nowcasts.loc[v_new, "news"] = news
            nowcasts.loc[v_new, "new"] = new_cast
            df = table.dropna().reset_index()
            df = df.rename(columns={-1: "Date"})
            df.loc[:, "Date"] = v_new
            nowcastv = nowcastv.append(df, ignore_index=True)
            print(v_new[-5:], end=" ")

        except:
            if os.path.exists(f"{RTF}/{v_old}{suffix}_predicted.csv"):
                shutil.copy(f"{RTF}/{v_old}{suffix}_predicted.csv", f"{RTF}/{v_new}{suffix}_predicted.csv")

            nowcasts.loc[v_new, "old"] = nowcasts.loc[v_old, "new"]
            nowcasts.loc[v_new, "rev"] = 0
            nowcasts.loc[v_new, "news"] = 0
            nowcasts.loc[v_new, "new"] = nowcasts.loc[v_old, "new"]
            print(f'\n{v_old} values and rtf file are copied')

    nowcasts.index = pd.to_datetime(nowcasts.index)
    nowcastv = nowcastv.set_index("Date")
    nowcastv = nowcastv.round(decimals=3)
    nowcastv.index = pd.to_datetime(nowcastv.index)

    nowcasts.to_pickle(f'{RTF}/nowcasts{suffix}.pkl')
    nowcastv.to_pickle(f'{RTF}/nowcastv{suffix}.pkl')
    print(f"nowcasts and nowcastv are pickled in {RTF}")

    dfm_rtf, dfm_rtf_detail = decomp_to_rtf(tq, vintages, nowcasts, nowcastv, specfile, RTF)
    print('dfm_rtf and dfm_rtf_detail are generated')

    dfm_rtf.to_pickle(f'{RTF}/dfm{suffix}_rtf.pkl')
    dfm_rtf_detail.to_pickle(f'{RTF}/dfm{suffix}_rtf_detail.pkl')
    print(f"dfm_rtf is pickled as {RTF}/dfm{suffix}_rtf.pkl")

    return dfm_rtf


def predict(X, ER):
    """
    para_const()    Implements Kalman filter for "News_DFM.m" without lag

    Description:
        This procedure smooths and fills in missing data for a given data matrix X.
        In contrast to runKF(), this function is used when model parameters are already estimated.

    Input
        X  : Data matrix.
        ER : Parameters from the dynamic factor model.

    Output
        X_sm: Smoothed data matrix

    Kalman filter with specified paramaters written for "MAXIMUM LIKELIHOOD ESTIMATION OF FACTOR MODELS
    ON DATA SETS WITH ARBITRARY PATTERN OF MISSING DATA." by Marta Banbura and Michele Modugno
    """

    ## Set model parameters and data preparation
    X_mean = ER.mean
    X_std = ER.std

    # Standardise x by means and s.e. from DFM module
    Y = ((X - X_mean) / X_std).T.to_numpy()

    # Set index/columns for states, periods, and periods X states
    states = pd.MultiIndex.from_tuples(ER.M.A.index)
    ps = X.index
    ps_0 = pd.date_range(ps[0] - MonthEnd(), ps[-1], freq=ps.freq)

    ps_states = MI.from_tuples([(i, j, p, q) for i in ps for j, p, q in states.to_flat_index()])
    ps_0_states = MI.from_tuples([(i, j, p, q) for i in ps_0 for j, p, q in states.to_flat_index()])

    ## Apply Kalman filter and smoother
    # See runKF() for details about FIS and SKF
    A, C, Q, R, Z_0, V_0 = dfm_copy(ER.M)

    Zm, ZmU, Vm, VmU, loglik, k_t = SKF(Y, A, C, Q, R, Z_0, V_0)  # Kalman filter
    ZmT, VmT, VmT_1 = FIS(A, Zm, ZmU, Vm, VmU, loglik, k_t)  # Fixed interval smoother

    # in FRBNY matlab code:
    # Sf = SKF(Y, A, C, Q, R, Z_0, V_0);  # Kalman filter
    # Ss = FIS(A, Sf);                    # Smoothing step

    ## Calculate parameter output
    Zsmooth = ZmT.copy()  # Smoothed factors

    # Prepare data for output
    F = pd.DataFrame(Zsmooth[1:, :], index=ps, columns=states).copy()

    X_sm = F.dot(ER.M.C.T) * X_std + X_mean  # Standardized to unstandardized

    return X_sm


def update_nowcast(ER, target, v_old, v_new, specfile, data, print_detail=True):
    """Run nowcast for two vintages v_old and v_new and decompose the difference by revision and new.

    Input
        ER          : DFM result
        target      : target variable and quarter for nowcasting
        v_old       : previous week
        v_new       : current week
        print_detail: default=True


    Output
        old        :   previous week projection
        rev        : + data revision effect
        news       : + news effect
        new        : = current week projection
        news_table : data to be written as shared file
    """

    spec, blocks = ER.spec, ER.blocks

    X_old, mean, std, normalized, X_data_old = load_data(f"{data}/{v_old}.xlsx", specfile, sample_start)
    X_new, mean, std, normalized, X_data_new = load_data(f"{data}/{v_new}.xlsx", specfile, sample_start)

    fps = pd.date_range(X_new.index[0], X_new.index[-1] + MonthEnd(12), freq=X_new.index.freq)

    X_new = X_new.reindex(fps)
    X_old = X_old.reindex(fps)

    # Update nowcast for target variable 'series' (i) at horizon 'period' (t)
    #   > Relate nowcast update into news from data releases:
    #     a. Compute the impact from data revisions
    #     b. Compute the impact from new data releases
    X_rev = X_new.copy()
    X_rev[X_old.isnull()] = np.nan

    # update_nowcast(X_old,X_new,Time,Spec,ER,series,period,v_old,v_new);
    U0 = News_DFM(X_old, X_rev, ER, target)
    # Compute impact from data revisions
    U1 = News_DFM(X_rev, X_new, ER, target)
    # Compute impact from data releases
    # UpdatedResult = namedtuple('UpdatedResult', 'o n N A F W I M') : New_DFM의 return인 namedtuple임
    """
    Input
        X_old  :  Old data matrix (old vintage)
        X_new  :  New data matrix (new vintage)
        ER     :  DFM() output results (see DFM for more details)
        target :

   Output: UpdatedResult(y_old, y_new, News, A, F, W, innov, updated_month)
        y_old        : Old nowcast
        y_new        : New nowcast
        News         : News for each data series
        A            : Observed series release values
        F            : Forecasted series values
        W            : News weight
        updated_month: pd.Series that displays which variables are newly imported for which month
        innov        : Difference between observed and predicted series values ("innovation")
    """

    if len(U1.F.dropna().index) == 0:
        # Only display table output if a forecast is made
        print("\nNo forecast was made...", end=" ")
    else:
        impact_revisions = U1.o - U0.o
        # Impact from revisions
        news = U1.A - U1.F.values
        # News from releases
        impact_releases = U1.W.mul(news.values, axis=0)
        # Impact of releases
        impact_releases.columns = ["Impact"]

        # Store results
        news_table = pd.concat([U1.F, U1.A, U1.W, impact_releases], axis=1)
        # concat에서 axis=1이면 가로로 붙이는 것

        if print_detail:
            # Display output
            print("\nNowcast Update: {:}  ".format(v_new))
            print("Nowcast for {:} ({:}), {:} \n\n".format(target.v,
                                                           spec.UnitsTransformed.loc[target.v],
                                                           target.t,
                                                          )
                 )  # target.t에는 날짜가 들어가 있음
            # Display the impact decomposition
            print("Nowcast Impact Decomposition")
            print("Note: The displayed output is subject to rounding error \n")
            print("{:>18} nowcast:   {:20,.2f}".format(v_old, U0.o))
            print("Impact from data revisions:                  {:5,.2f}".format(impact_revisions))
            print(" Impact from data releases:                  {:5,.2f}".format(news_table.Impact.sum()))
            print("                                          +_________")
            print("              Total impact:                  {:5,.2f}".format(impact_revisions + news_table.Impact.sum()))
            print("{:>18} nowcast:               {:20,.2f}\n".format(v_new, U1.n))

            # Display the table output
            print("\n  Nowcast Detail Table \n")
            print(news_table.dropna(how="all", axis=0).applymap(dp4))

            """
            Nowcast Update: 2018-03-23
            Nowcast for N_gdp (raw), 2018-03-31

            Nowcast Impact Decomposition
            Note: The displayed output is subject to rounding error

            2018-03-16 nowcast:                               0.95
            Impact from data revisions:                       0.00
            Impact from data releases:                        0.04
                                                          +_________
            Total impact:                                     0.04
            2018-03-23 nowcast:                               0.99


              Nowcast Detail Table

                         Forecast  Actual  Weight  Impact
            SeriesID
            P_ppi     -0.0138  0.4194  0.0917  0.0397
            """

        return (U0.o, impact_revisions, news_table.Impact.sum(), U1.n, news_table, U1.df_n,)


def News_DFM(X_old, X_new, ER, target):
    """
    News_DFM()    Calculates changes in news

    Syntax:

    Description:
        News DFM() inputs two datasets, DFM parameters, target time index, and
        target variable index. The function then produces Nowcast updates and
        decomposes the changes into news.

    Input
        X_old  :  Old data matrix (old vintage)
        X_new  :  New data matrix (new vintage)
        ER     :  DFM() output results (see DFM for more details)
        target :

    Output: UpdatedResult(y_old, y_new, News, A, F, W, innov, updated_month)
        y_old        : Old nowcast
        y_new        : New nowcast
        News         : News for each data series
        A            : Observed series release values
        F            : Forecasted series values
        W            : News weight
        updated_month: pd.Series that displays which variables are newly imported for which month
        innov        : Difference between observed and predicted series values ("innovation")
        df_n은 X_new에서 NaN만 DFM으로 forecasted된 variable을 채워넣음
    """

    t_var, t_month = target
    t_month = pd.to_datetime(t_month)

    date_diff = lambda fd, md: (fd.year - md.year) * 12 + (fd.month - md.month)

    ps = X_old.index
    ps_0 = pd.date_range(ps[0] - MonthEnd(), ps[-1], freq=ps.freq)
    ps_inv = pd.Series(range(len(ps)), index=ps)

    ## Initialize variables
    r = ER.M.C.shape[1]  # r: no. of states
    T, N = X_new.shape  # T: no. of observations, N: no. of observed variables

    y_old = pd.Series(np.nan, index=[t_var])
    y_new = pd.Series(np.nan, index=[t_var])

    actual = pd.Series(np.nan, index=X_new.columns)
    forecast = pd.Series(np.nan, index=X_new.columns)

    # Initialize news vector (will store news for each series)
    News = pd.DataFrame(np.nan, index=X_new.columns, columns=[t_var])
    weight = pd.DataFrame(np.nan, index=X_new.columns, columns=[t_var])

    ## NO FORECAST CASE: value for t_var at t_month exists
    if ~np.isnan(X_new.loc[t_month, t_var]):

        Sf = para_const(X_old, ER, 0)  # Apply Kalman filter for old data

        for i in [t_var]:  # Loop for each target variable
            # (Observed value) - (predicted value)
            News.loc[i] = X_new.loc[t_month, i] - Sf.X_sm.loc[t_month, i]

            # Set predicted and observed y values
            y_old.loc[i] = Sf.X_sm.loc[t_month, i]
            y_new.loc[i] = X_new.loc[t_month, i]

        # Forecast-related output set to empty
        innov = pd.Series(float)
        updated_month = pd.Series(float)

    ## FORECAST CASE (these are broken down into (A) and (B))
    else:
        # Initialize series mean/standard deviation respectively
        X_mean = ER.mean
        X_std = ER.std

        # Calculate indicators for missing values (1 if missing, 0 otherwise)
        miss_old = X_old.isnull()
        miss_new = X_new.isnull()

        # Indicator for missing--combine above information to single matrix where:
        # (i) -1: Value is in the old data, but missing in new data
        # (ii) 1: Value is in the new data, but missing in old data
        # (iii) 0: Values are missing from/available in both datasets
        i_miss = miss_old.astype(int) - miss_new.astype(int)
        i_miss[i_miss != 1] = np.nan
        #############################################################
        ## In each update, one variable has only one new release!!!
        ## This means that updated_month has no duplicates
        #############################################################
        updated_month = i_miss.stack().dropna().reset_index().set_index("SeriesID").iloc[:, 0]
        innov = pd.Series(np.nan, index=updated_month.index)

        ## FORECAST SUBCASE (A): NO NEW INFORMATION
        if len(updated_month.index) == 0:
            # Fill in missing variables using a Kalman filter
            S0 = para_const(X_old, ER, 0)
            S1 = para_const(X_new, ER, 0)

            # Set predicted and observed y values. New y value is set to old
            y_old = S0.X_sm.loc[t_month, t_var].copy()
            y_new = y_old.copy()
            # y_new = ER_new.X_sm(t_month,t_var);

            # Forecast-related output set to empty
            innov = pd.Series(float)
            updated_month = pd.Series(float)

        # ----------------------------------------------------------------------
        #     v_miss=[1:size(X_new,2)]';
        #     t_miss=t_miss(1)*ones(size(X_new,2),1);
        # ----------------------------------------------------------------------
        ## FORECAST SUBCASE (B): NEW INFORMATION
        else:
            # Difference between forecast time and new data time
            lag = np.array([date_diff(t_month, updated_month[i]) for i in updated_month.index])

            # Gives biggest time interval between forecast and new data
            k = max(np.append(abs(lag), max(lag) - min(lag)))

            C = ER.M.C  # Observation matrix
            R = ER.M.R.T  # Covariance for observation matrix residuals

            # Smooth old dataset
            S0 = para_const(X_old, ER, k)

            # Smooth new dataset
            S1 = para_const(X_new, ER, 0)

            # Subset for target variable and forecast time
            y_old = S0.X_sm.loc[t_month, t_var]
            y_new = S1.X_sm.loc[t_month, t_var]

            P = S0.P.loc[:, idx[ps_0[1:]]]
            P1 = np.empty((r, 0))  # Initialize projection onto updates

            # Cycle through total number of updates
            for i in updated_month.index:
                h = abs(date_diff(t_month, updated_month[i]))
                m = max(t_month, updated_month[i])

                # If location of update is later than the forecasting date
                if updated_month[i] > t_month:
                    Pp = S0.Plag[h][ps_inv[m], :, :]  # P(1:r,h*r+1:h*r+r,m)';
                else:
                    Pp = S0.Plag[h][ps_inv[m], :, :].T  # P(1:r,h*r+1:h*r+r,m);
                P1 = np.append(P1, Pp.dot(C.loc[[i], :].T), axis=1)  # Projection on updates

            for i in updated_month.index:
                # Standardize predicted and observed values
                X_new_norm = (X_new.loc[updated_month[i], i] - X_mean[i]) / X_std[i]
                X_sm_norm = (S0.X_sm.loc[updated_month[i], i] - X_mean[i]) / X_std[i]

                # Innovation: Gives [observed] data - [predicted data]
                innov[i] = X_new_norm - X_sm_norm

            ins = innov.shape[0]
            WW = pd.DataFrame(np.zeros((ins, ins)), index=updated_month.index, columns=updated_month.index,)
            P2 = np.empty((0, ins))

            # Gives non-standardized series weights
            for i, iv in enumerate(updated_month.index):

                p2 = np.empty((1, 0))
                for j, jv in enumerate(updated_month.index):
                    h = abs(lag[i] - lag[j])
                    m = max(updated_month[iv], updated_month[jv])

                    if updated_month[jv] > updated_month[iv]:
                        Pp = S0.Plag[h][ps_inv[m], :, :]  # P(1:r,h*r+1:(h+1)*r,m)';
                    else:
                        Pp = S0.Plag[h][ps_inv[m], :, :].T  # P(1:r,h*r+1:(h+1)*r,m);

                    if (iv == jv) & (updated_month[iv] != updated_month[jv]):  # this never happens!
                        WW.loc[iv, jv] = 0
                    else:
                        WW.loc[iv, jv] = R.loc[iv, jv]

                    p2 = np.append(p2, np.array([[C.loc[iv, :].dot(Pp).dot(C.loc[jv, :].T) + WW.loc[iv, jv]]]), axis=1,)
                P2 = np.append(P2, p2, axis=0)

            totnews = pd.Series(0, index=[t_var])
            temp = pd.DataFrame(0, index=updated_month.index, columns=[t_var])
            gain = pd.DataFrame(0, index=updated_month.index, columns=[t_var])

            for i in [t_var]:  # loop on t_var
                # Convert to real units (unstadardized data)
                totnews[i] = X_std[i] * C.loc[i, :].dot(P1).dot(inv(P2)).dot(innov.to_numpy().reshape((ins, 1))).squeeze()
                temp.loc[:, i] = X_std[i] * C.loc[i, :].dot(P1).dot(inv(P2)) * innov.to_numpy()
                gain.loc[:, i] = X_std[i] * C.loc[i, :].dot(P1).dot(inv(P2))

            # Fill in output values
            # If a variable has two or more new releases, the eq. for News should be changed to
            # ""News(t_miss(i)-min(t_miss)+1, v_miss(i), j) = temp(1,i,j)""
            # See FRBNY matlab code
            for i in updated_month.index:
                actual[i] = X_new.loc[updated_month[i], i]
                forecast[i] = S0.X_sm.loc[updated_month[i], i]

                for j in [t_var]:
                    News.loc[i, j] = temp.loc[i, j]
                    weight.loc[i, j] = gain.loc[i, j] / X_std[i]

    F = forecast.to_frame("Forecast")
    A = actual.to_frame("Actual")
    W = weight.iloc[:, 0].to_frame("Weight")
    df_n = X_new.fillna(S1.X_sm)

    return UpdatedResult(y_old, y_new, News, A, F, W, innov, updated_month, df_n)


def para_const(X, ER, lag):
    """
    para_const()    Implements Kalman filter for "News_DFM.m"

    Syntax:
        ER = para_const(X,P,lag)

    Description:
        para_const() implements the Kalman filter for the news calculation step. This procedure
        smooths and fills in missing data for a given data matrix X. In contrast to runKF(),
        this function is used when model parameters are already estimated.

    Input
        X  : Data matrix.
        ER : Parameters from the dynamic factor model.
        lag: Number of lags

    Output
        Plag: Smoothed factor covariance for transition matrix
        P   :    Smoothed factor covariance matrix
        X_sm: Smoothed data matrix
        F   :    Smoothed factors

    Kalman filter with specified paramaters written for "MAXIMUM LIKELIHOOD ESTIMATION OF FACTOR MODELS
    ON DATA SETS WITH ARBITRARY PATTERN OF MISSING DATA." by Marta Banbura and Michele Modugno
    """

    ## Set model parameters and data preparation
    X_mean = ER.mean
    X_std = ER.std

    # Standardise x by means and s.e. from DFM module
    Y = ((X - X_mean) / X_std).T.to_numpy()

    # Set index/columns for states, periods, and periods X states
    states = pd.MultiIndex.from_tuples(ER.M.A.index)
    ps = X.index
    ps_0 = pd.date_range(ps[0] - MonthEnd(), ps[-1], freq=ps.freq)

    ps_states = MI.from_tuples([(i, j, p, q) for i in ps for j, p, q in states.to_flat_index()])
    ps_0_states = MI.from_tuples([(i, j, p, q) for i in ps_0 for j, p, q in states.to_flat_index()])

    ## Apply Kalman filter and smoother
    # See runKF() for details about FIS and SKF
    A, C, Q, R, Z_0, V_0 = dfm_copy(ER.M)

    Zm, ZmU, Vm, VmU, loglik, k_t = SKF(Y, A, C, Q, R, Z_0, V_0)  # Kalman filter
    ZmT, VmT, VmT_1 = FIS(A, Zm, ZmU, Vm, VmU, loglik, k_t)  # Fixed interval smoother

    # in FRBNY matlab code:
    # Sf = SKF(Y, A, C, Q, R, Z_0, V_0);  # Kalman filter
    # Ss = FIS(A, Sf);                    # Smoothing step

    ## Calculate parameter output
    Vs = VmT[1:, :, :].copy()  # ?? not used... Smoothed factor covariance for transition matrix
    Vf = VmU[1:, :, :].copy()  # Filtered factor posterior covariance
    Zsmooth = ZmT.copy()  # Smoothed factors
    Vsmooth = VmT.copy()  # Smoothed covariance values

    Plag = {0: Vs}
    for jk in range(1, lag + 1):
        pl = np.zeros(Vs.shape)
        for pl_j, Vf_j, Plag_j in zip(pl[-(pl.shape[0] - lag) :], Vf[-(pl.shape[0] - lag) - jk : -jk], Plag[jk - 1][-(pl.shape[0] - lag) :],):
            As = Vf_j.dot(A.T).dot(pinv(A.dot(Vf_j).dot(A.T) + Q))
            pl_j[:, :] = As.dot(Plag_j)
        Plag[jk] = pl

    # Prepare data for output
    F = pd.DataFrame(Zsmooth[1:, :], index=ps, columns=states).copy()

    Vsmooth = np.moveaxis(Vsmooth, 0, -1).reshape(len(states), len(ps_0_states), order="F")
    P = pd.DataFrame(Vsmooth, index=states, columns=ps_0_states)

    X_sm = F.dot(ER.M.C.T) * X_std + X_mean  # Standardized to unstandardized
    # X_nsm = F.dot(ER.M.C.T)

    # --------------------------------------------------------------------------
    #   Loading the structure with the results
    # --------------------------------------------------------------------------
    return SmoothedFactor(Plag, P, X_sm, F)


def decomp_to_rtf(tq, vintages, nowcasts, nowcastv, specfile, RTF):
    """DFM 요인분해 결과를 rtf 데이터프레임으로 변환"""

    spec, _ = load_spec(specfile)

    nowcastv1 = nowcastv.reset_index().merge(spec.reset_index()[['SeriesID', 'Category']],
                                             on='SeriesID', how='left')
    nowcastv1 = nowcastv1.set_index('Date')

    # 매 전망시점의 변동요인 설명을 위한 변수명의 한글이름 컬럼 추가하여 저장
    # [TODO] add kind, target cols
    nowcastv3 = nowcastv.reset_index().merge(spec.reset_index()[['SeriesID', 'SeriesName', 'Category']],
                                             on='SeriesID', how='left')
    nowcastv3 = nowcastv3.set_index('Date')
    nowcastv3 = nowcastv3[['SeriesID', 'SeriesName', 'Category', 'Forecast', 'Actual', 'Weight', 'Impact']]

    dfm_rtf_detail = nowcastv3.copy()

    nowcastv2 = nowcastv1.groupby(['Date', 'Category']).sum()
    nowcastv2 = nowcastv2.unstack().Impact
    nowcastv2.index.name = None
    nowcastv2 = nowcastv2.join(nowcasts.new.dropna())
    nowcastv2 = nowcastv2.rename(columns={"new": "dfm"})

    # 주차가 시작되면서 발생한 변동요인에 대해서만 컬럼을 추가하기 때문에, 전망 초기에는 어떤 컬럼이 없을 수 있고
    # 따라서 plot 함수 동작시 오류가 날 수 있으므로, 매주마다 없는 컬럼을 체크해서 빈 값을 넣어주는 것임
    # nowcastv2에서 -1를 한 이유는, nowcasting~잠정치까지는 위에서 hard code로 넣어준 것이라 반드시 있기 때문에 제외가능
    for e in list(set(spec['Category']) - set(nowcastv2.columns[:-1])):
            nowcastv2[e] = np.nan

    # 위에서 추가된 컬럼이 있다면 이것은 nowcasting~잠정치가 나온 후에 추가되기 때문에,
    # 최종적으로 컬럼 순서를 조정하여, LSTM 결과값을 쓰는 코드가 오류가 나지 않도록 함
    model_cols = list(set(spec['Category'])) + ['dfm']
    nowcastv2 = nowcastv2[model_cols]

    # For a vintage date, if 'old' and 'new' values of nowcasts in the date are NaNs and the vintage data exists,
    # set 'nowcasting' value of nowcastsv2 in the date to the values in the previous vintage date.
    for vdate in nowcastv2.index:
        if nowcasts.loc[vdate, ['old', 'new']].isna().all():
            if os.path.exists("./input/mdata/" + vdate.strftime('%Y-%m-%d') + ".xlsx"):
                nowcastv2.loc[vdate, "dfm"] = nowcastv2.shift().loc[vdate, "dfm"]
            else:
                nowcastv2.loc[vdate, "dfm"] = np.nan

    df = nowcastv2.sort_index().copy()
    df.index = df.index.strftime('%Y-%m-%d')
    df = df.reindex(vintages)
    df.loc[vintages[0], 'dfm'] = nowcasts.loc[vintages[0], 'new']
    df.loc[:, 'dfm'] = df.loc[:, 'dfm'].fillna(method='ffill')
    df.index = pd.to_datetime(df.index)
    dfm_rtf = df.copy()

    return dfm_rtf, dfm_rtf_detail



In [ ]:

# df.loc[52, 'Series ID'] = 'KOSIS-101_DT_1J20007-M-T-QB-14101_ausgabe'
# df.loc[52, 'Series Name'] = '농산물 및 석유류제외지수(2020＝100) (월,분기,년 1975.01~2023.01)-월-소비자물가지수-농산물및석유류제외지수-2020＝100'
# df.loc[52, 'Dataset ID'] = '101_DT_1J20007'
# df.loc[52, 'ID'] = 'P_core1'

# df.loc[66, 'Series ID'] = 'KOSIS-101_DT_1J20009-M-T-DB-14101_ausgabe'
# df.loc[66, 'Series Name'] = '식료품 및 에너지제외지수(2020＝100) (월,분기,년 1990.01~2023.01)-월-소비자물가지수-식료품 및 에너지제외 지수-2020＝100'
# df.loc[66, 'Dataset ID'] = '101_DT_1J20009'
# df.loc[66, 'ID'] = 'P_core2'

# df.loc[59, 'Series ID'] = 'NECOS-511U002-M-FMAB-99988'
# df.loc[59, 'Series Name'] = '소비자동향조사(전국, 월, 2008.9~)-월-현재경기판단CSI-전체'
# df.loc[59, 'Dataset ID'] = '511U002'
# df.loc[59, 'ID'] = 'S_cb'

# df.loc[60, 'Series ID'] = 'NECOS-511U002-M-FME-99988'
# df.loc[60, 'Series Name'] = '소비자동향조사(전국, 월, 2008.9~)-월-소비자심리지수-전체'
# df.loc[60, 'Dataset ID'] = '511U002'
# df.loc[60, 'ID'] = 'S_cs'

# df.loc[65, 'Series ID'] = 'NECOS-511U002-M-FMBB-99988'
# df.loc[65, 'Series Name'] = '소비자동향조사(전국, 월, 2008.9~)-월-향후경기전망CSI-전체'
# df.loc[65, 'Dataset ID'] = '511U002'
# df.loc[65, 'ID'] = 'S_fc'

# df.to_excel('input/meta_202303.xlsx')

# Z = pd.read_excel(f"input/mdata/2023-03-07.xlsx", sheet_name="data", index_col=0)

# updated_variables = ['P_core1', 'P_core2', 'S_cb', 'S_cs', 'S_fc']

# Z = pd.read_excel(f"input/mdata/2023-03-07.xlsx", index_col=0)

# for file in sorted(files):
#     if file[-3:] == 'csv':
#         continue

#     import_date = pd.to_datetime(file[12:22])
#     last_month = (import_date + MonthEnd()).strftime("%Y-%m-%d")

#     df = pd.read_excel(file, index_col=0)
#     df.loc[:last_month, updated_variables] = Z.loc[:last_month, updated_variables]
#     df.to_excel(file)

# Plot and evaluation

In [ ]:
def plot_nowcasts(tq, dfm_suffix, lstm_suffix, ax=None, fontsize=14):

    if not os.path.isdir("./plot"):
        os.mkdir("./plot")
        print(f"\nDirectory ./plot was created.")

    df = combine_dfm_lstm_gdp(tq, dfm_suffix, lstm_suffix)
    try:
        df['mean'] = df[['dfm', 'lstm']].mean(axis=1)
    except:
        pass

    forecasting_week = df.dropna(how='all', axis=0).index[-1].strftime('%Y-%m-%d')

    ulim = df.max().max()
    llim = df.min().min()
    h = 0.1 * (ulim - llim)
    ticklabels = [t.strftime("%b\n%d") for t in df.index]

    df.index = np.arange(len(df.index))
    df['dfm'].plot(ax=ax, color=(0.01, 0.09, 0.44), lw=5*(fontsize/16), marker="o", markersize=10*(fontsize/16), label="DFM", alpha=0.8)
    #ax.plot(df.index, df['dfm'], color=(0.01, 0.09, 0.44), lw=5*(fontsize/16), marker="o", markersize=10*(fontsize/16), label="DFM", alpha=0.8)
    ax.plot(df.index, df.fillna(df.mean())['dfm'], alpha=0)

    try:
        ax.plot(df.index, df['lstm'], color="purple", lw=5*(fontsize/16), marker="o", markersize=10*(fontsize/16), label="LSTM", alpha=0.8)
        ax.plot(df.index, df['mean'], color="gray", lw=5*(fontsize/16), marker="o", markersize=10*(fontsize/16), label="ENSEMBLE", alpha=0.9)
    except:
        pass

    try:
        ax.scatter(df.index, df["flash"], color=(35 / 256, 55 / 256, 59 / 256), marker="s", s=250*(fontsize/16), label="flash",)
        ax.scatter(df.index,  df["provisional"], color="red", marker="*", s=250*(fontsize/16), label="provisional")
    except:
        pass

    try:
        df = df.rename(columns = {'Retail and Consumption':'Retail', 'International Trade':'Intl. Trade',
                                  'Manufacturing':'MFG', 'National Accounts':'NA'})
        decomp = df[['Prices', 'Intl. Trade', 'Surveys', 'MFG', 'Labor', 'Retail', 'NA']]
        decomp.plot.bar(stacked=True, color=bcolors, width=0.8, ax=ax)
    except:
        pass

    ax.set_ylim(llim - h, ulim + h)
    ax.set_xticklabels(ticklabels[::2], rotation=0)
    leg1 = ax.legend(ncol=3, bbox_to_anchor=(-0.05, -0.12), loc="upper left", frameon=False, fontsize=fontsize)
    ax.grid(axis="both", color=(200 / 256, 200 / 256, 200 / 256), linestyle="--", alpha=0.9)
    ax.axhline(0, 0, 1, color="black", linestyle="-", linewidth="1")
    ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.tick_params(axis="x", labelsize=fontsize)
    ax.tick_params(axis="y", labelsize=fontsize)
    ax.text(0., 1.01, f"last updated: {forecasting_week}", fontsize=fontsize, transform=ax.transAxes)


def combine_dfm_lstm_gdp(tq, dfm_suffix='_base', lstm_suffix='_base'):

    RTF = f"./rtf/{str(tq)}"
    # lstm_RTF = f"/home/work/newtech2/nowcasting/rtf/{str(tq)}"

    dfm_rtf = pd.read_pickle(f'{RTF}/dfm{dfm_suffix}_rtf.pkl')
    rtf = dfm_rtf.copy()

    try:
        lstm_rtf = pd.read_pickle(f'{RTF}/lstm{lstm_suffix}_rtf.pkl')
        lstm_rtf.index = pd.to_datetime(lstm_rtf.index)
        rtf.loc[:, 'lstm'] = lstm_rtf['mean']
        rtf.loc[:, 'lstm_med'] = lstm_rtf['median']
        rtf.loc[:, 'mean'] = rtf.loc[:, ['dfm', 'lstm']].mean(axis=1)
    except:
        pass

    GDP = pd.read_pickle('./input/GDP_releases.pkl')
    if not(np.isnan(GDP.loc[str(tq)]).all()):  # if flash estimate exists,
        last_date = rtf.index[-1] + pd.DateOffset(7)
        rtf = rtf.append(pd.DataFrame(index=[last_date]))
        rtf.loc[last_date, ['flash', 'provisional']] = GDP.loc[str(tq)]
    else:
        _, _, _, v = get_target_vintages(pd.Period(tq), display_vintages=False)
        v = pd.to_datetime(v)
        rtf = rtf.reindex(v)

    return rtf


def evaluate_models(start, end, dfm_suffix, lstm_suffix):
    """
    input:
            start    : the first quarter of the periods from which the evaluation is performed
            end      : the last quarter of the periods from which the evaluation is performed
    output:
            rmse     : RMSE of DFM and LSTM (mean, median, mode) weekly estimates during the evaluation periods
            mae      : MAE of DFM and LSTM (mean, median, mode) weekly estimates during the evaluation periods
    """
    ERROR = pd.DataFrame()
    periods = pd.period_range(start, end, freq='Q')

    for tq in periods:

        rtf = combine_dfm_lstm_gdp(tq, dfm_suffix, lstm_suffix)

        target = rtf.loc[:, ['flash', 'provisional']]
        target = target.dropna(axis=0, how='all')

        estimate = rtf[['dfm', 'lstm', 'lstm_med']].dropna()
        estimate['mean'] = estimate[['dfm', 'lstm']].mean(axis=1)
        error_flash = estimate - target['flash'].values[0]
        error_provisional = estimate - target['provisional'].values[0]

        error = pd.concat([error_flash, error_provisional], axis=1, keys=['flash', 'provisional'])
        error.index = [i for i in range(-len(error_flash.index), 0)]
        error = pd.concat([error], axis=0, keys=[tq])

        ERROR = pd.concat([ERROR, error], axis=0)

    ERROR = ERROR.reorder_levels([1, 0]).sort_index()
    rmse = ERROR.groupby(level=0, axis=0).apply(lambda x: np.mean(x**2)**(0.5))
    mae = ERROR.groupby(level=0, axis=0).apply(lambda x: np.mean(np.abs(x)))

    var = ['dfm', 'lstm', 'mean']
    var_legends = ['DFM', 'LSTM', 'ENSEMBLE']

    lcolors = [(85/256, 142/256, 213/256), (110/256, 110/256, 110/256),
               (175/256, 175/256, 105/256), (250/256, 192/256, 145/256),
               (150/256, 115/256, 170/256)]

    plt.rcParams["xtick.labelsize"] = 14
    plt.rcParams["ytick.labelsize"] = 14

    loss = ['RMSE', 'MAE']
    targets = ['flash', 'provisional']

    figsize = (40, 7.5)
    fig, ax = plt.subplots(1, 4, figsize=figsize)

    rmse.loc[-19:, targets[0]][var].plot(ax=ax[0], lw=3, marker="o", markersize=8, color=lcolors)
    mae.loc[-19:, targets[0]][var].plot(ax=ax[1], lw=3, marker="o", markersize=8, color=lcolors)
    rmse.loc[-19:, targets[1]][var].plot(ax=ax[2], lw=3, marker="o", markersize=8, color=lcolors)
    mae.loc[-19:, targets[1]][var].plot(ax=ax[3], lw=3, marker="o", markersize=8, color=lcolors)

    for i, a in enumerate(ax.ravel()):
        a.legend(var_legends, ncol=1, frameon=False, fontsize=14,)
        a.grid(axis="both", color=(200 / 256, 200 / 256, 200 / 256), linestyle="--", lw=2, alpha=0.9)
        a.xaxis.set_major_locator(ticker.MultipleLocator(2))
        a.tick_params(axis="x", labelsize=14)

    ax[0].set_title(f"{loss[0]} against flash : {start} - {end}", fontsize=16, y=1.01)
    ax[1].set_title(f"{loss[1]} against flash : {start} - {end}", fontsize=16, y=1.01)
    ax[2].set_title(f"{loss[0]} against prov. : {start} - {end}", fontsize=16, y=1.01)
    ax[3].set_title(f"{loss[1]} against prov. : {start} - {end}", fontsize=16, y=1.01)

    ax[0].set_ylim([0.2, 1.6])
    ax[1].set_ylim([0.1, 1.1])
    ax[2].set_ylim([0.2, 1.6])
    ax[3].set_ylim([0.1, 1.1])

    rmse_mae = pd.concat([rmse, mae], axis=1, keys=['rmse', 'mae'])
    #rmse_mae.to_excel(f"./plot/rmse_mae_{start}_{end}{suffix}.xlsx")

    return rmse_mae

# estimate

In [ ]:
specfile = "/content/drive/MyDrive/nowcasting/Spec_kim_34var_transformchange_named.xlsx"
forecasting_week = datetime.now().strftime('%Y-%m-%d') # the last date of the vintages

# DFM structure
spec, blocks = load_spec(specfile)
S = DFM_structure(blocks)  # r, p, Rcon, q

In [ ]:
S.r

,0
Global,1
Soft,1
Real,1
Labor,1


In [ ]:
S

DFMStructure(r=Global    1
Soft      1
Real      1
Labor     1
dtype: int64, p=1, Rcon=array([[ 2, -1,  0,  0,  0],
       [ 3,  0, -1,  0,  0],
       [ 2,  0,  0, -1,  0],
       [ 1,  0,  0,  0, -1]]), q=array([0., 0., 0., 0.]))

In [ ]:
blocks

,Global,Soft,Real,Labor
SeriesID,,,,
P_core1,1,0,0,0
P_core2,1,0,0,0
P_ppi,1,0,0,0
B_pi,1,0,1,0
S_as,1,1,0,0
S_ab,1,1,0,0
S_mu,1,1,0,0
S_mx,1,1,0,0
S_md,1,1,0,0


In [ ]:
spec

,SeriesName,Frequency,Transformation,Units,Category,sa,UnitsTransformed
SeriesID,,,,,,,
P_core1,소비자물가지수(농산물 석유류 제외),m,pch,2015＝100,Prices,0,Percent Change
P_core2,소비자물가지수(식료품 에너지 제외),m,pch,2015＝100,Prices,0,Percent Change
P_ppi,생산자물가지수,m,pch,2015=100,Prices,0,Percent Change
B_pi,수입물가지수,m,pch,2015=100,International Trade,0,Percent Change
S_as,전산업매출BSI,m,raw,NaN,Surveys,0,raw
S_ab,전산업업황BSI,m,raw,NaN,Surveys,0,raw
S_mu,제조업가동률BSI,m,raw,NaN,Surveys,0,raw
S_mx,제조업수출BSI,m,raw,NaN,Surveys,0,raw
S_md,제조업내수판매BSI,m,raw,NaN,Surveys,0,raw


In [ ]:
dfm_suffix = '_11' # old mdata and unnormalized N_gdp, 9 for 25 sample yrs

#input_data = '/content/nowcasting/mdata_updated'
input_data = '/content/nowcasting/mdata_updated'

if not os.path.isdir(f"/content/nowcasting/model"):
    os.mkdir(f"/content/nowcasting/model")

if not os.path.isdir(f"/content/nowcasting/model/DFM{dfm_suffix}"):
    os.mkdir(f"/content/nowcasting/model/DFM{dfm_suffix}")

In [ ]:
# pd.Period('2022Q4')
start = '2018Q1'
end = '2023Q2'

for tq in pd.period_range(start, end, freq='Q'):

    # SET vintages for estimation and nowcasting
    fvintage, lvintage, sample_start, vintages = get_target_vintages(tq, forecasting_week, sample_yrs=12, display_vintages=False)

    transformed, mean, std, trans_normalized, raw = load_data(f"{input_data}/{fvintage}.xlsx", specfile, sample_start)

    # parameters for EM loop: threshold, max_iter, gamma
    dfm_model = dfm(trans_normalized, mean, std, S, spec, blocks, threshold=0.002, max_iter=2000, gamma=0.5, suffix=dfm_suffix)


DFM estimation for 2018Q1 with obs. 2006-02 - 2017-11 with initial loglik 507.49
   0, 00:41:49,    3 sec, loglik = 579.82,    loglik - pre_loglik = 72.33
  30, 00:42:29,   39 sec, loglik = 682.06,    loglik - pre_loglik = 0.21
  60, 00:43:12,   42 sec, loglik = 694.72,    loglik - pre_loglik = 0.38
  90, 00:43:52,   40 sec, loglik = 700.98,    loglik - pre_loglik = 0.07
 120, 00:44:32,   40 sec, loglik = 703.13,    loglik - pre_loglik = 0.08
 150, 00:45:13,   40 sec, loglik = 704.60,    loglik - pre_loglik = 0.03
 180, 00:45:55,   42 sec, loglik = 705.24,    loglik - pre_loglik = 0.02
 210, 00:46:35,   39 sec, loglik = 705.68,    loglik - pre_loglik = 0.01
 240, 00:47:15,   40 sec, loglik = 705.96,    loglik - pre_loglik = 0.01
 270, 00:47:55,   40 sec, loglik = 706.15,    loglik - pre_loglik = 0.01
 300, 00:48:38,   42 sec, loglik = 706.30,    loglik - pre_loglik = 0.00
 330, 00:49:18,   40 sec, loglik = 706.39,    loglik - pre_loglik = 0.00
Loop ended with loglik 706.44 in 351 iter

In [ ]:
!zip -r dfm_11.zip /content/nowcasting/model/DFM_11/

  adding: content/nowcasting/model/DFM_11/ (stored 0%)
  adding: content/nowcasting/model/DFM_11/2019Q2.model (deflated 56%)
  adding: content/nowcasting/model/DFM_11/2023Q1.model (deflated 55%)
  adding: content/nowcasting/model/DFM_11/2022Q4.model (deflated 55%)
  adding: content/nowcasting/model/DFM_11/2023Q2.model (deflated 55%)
  adding: content/nowcasting/model/DFM_11/2018Q1.model (deflated 56%)
  adding: content/nowcasting/model/DFM_11/2019Q3.model (deflated 55%)
  adding: content/nowcasting/model/DFM_11/2020Q3.model (deflated 56%)
  adding: content/nowcasting/model/DFM_11/2021Q3.model (deflated 55%)
  adding: content/nowcasting/model/DFM_11/2022Q3.model (deflated 55%)
  adding: content/nowcasting/model/DFM_11/2021Q1.model (deflated 54%)
  adding: content/nowcasting/model/DFM_11/2019Q4.model (deflated 56%)
  adding: content/nowcasting/model/DFM_11/2018Q3.model (deflated 56%)
  adding: content/nowcasting/model/DFM_11/2022Q1.model (deflated 55%)
  adding: content/nowcasting/model/

In [ ]:
cp -a dfm_11.zip /content/drive/MyDrive/nowcasting/dfm_11_12yrs.zip